# DRIFT: Domain-Residual Rank Allocation for Parameter-Efficient Adaptation

Reproduces every experiment in the paper. Works on **Kaggle** and **Google Colab**.

**Kaggle:** in the right-hand panel set **Accelerator = GPU T4 x2** and **Internet = On**, then use *Save Version -> Save & Run All (Commit)* so it runs in the background. Each session stops starting new runs after `DEADLINE_HOURS`, so it always finishes inside Kaggle's 12-hour limit and saves `drift_results.zip`. To continue in a later session, upload that zip as a Kaggle Dataset (or a new version of it), attach it with *Add Input*, and run again: finished runs are restored and skipped.

**Colab:** *Runtime -> Change runtime type -> T4 GPU*, then *Run all*. Results are written straight to Google Drive (`MyDrive/drift_results/`), so a disconnect loses at most the run in progress: just *Run all* again.


In [ ]:
import os, sys, subprocess, time
SESSION_START = time.time()
ON_KAGGLE = os.path.exists('/kaggle/working')
ON_COLAB = 'google.colab' in sys.modules
WORK = '/kaggle/working' if ON_KAGGLE else '/content/drift'
os.makedirs(WORK, exist_ok=True)
os.chdir(WORK)
print('platform:', 'kaggle' if ON_KAGGLE else 'colab' if ON_COLAB else 'other',
      '| work dir:', WORK)
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total',
                      '--format=csv'], capture_output=True, text=True).stdout)
import torch
NGPU = torch.cuda.device_count()
print('torch', torch.__version__, '| GPUs:', NGPU)
assert NGPU > 0, 'No GPU: enable a GPU accelerator/runtime first.'


The harness implements every PEFT method itself, so the only requirements are torch, transformers, pandas/pyarrow, scikit-learn and scipy -- all preinstalled on Kaggle and Colab. We only install what is genuinely missing, rather than upgrading packages and risking the environment.

In [ ]:
import importlib
need = []
for mod, pkg in [('torch','torch'), ('transformers','transformers'),
                 ('pandas','pandas'), ('pyarrow','pyarrow'),
                 ('sklearn','scikit-learn'), ('scipy','scipy')]:
    try:
        importlib.import_module(mod)
    except ImportError:
        need.append(pkg)
print('missing:', need or 'nothing')
if need:
    subprocess.run([sys.executable,'-m','pip','install','-q',*need], check=True)
import transformers
print('transformers', transformers.__version__)


## Write the source tree

In [ ]:
import base64, json
os.makedirs(os.path.join(WORK, 'src'), exist_ok=True)
PAYLOAD = json.loads(r'''{"common.py": "IiIiU2hhcmVkIHV0aWxpdGllczogc2VlZGluZywgbWV0cmljcywgcGFyYW1ldGVyIGFjY291bnRpbmcsIHRpbWluZy4iIiIKaW1wb3J0IGpzb24sIG9zLCByYW5kb20sIHRpbWUsIGhhc2hsaWIKaW1wb3J0IG51bXB5IGFzIG5wCgpkZWYgc2V0X3NlZWQoc2VlZDogaW50KToKICAgIHJhbmRvbS5zZWVkKHNlZWQpOyBucC5yYW5kb20uc2VlZChzZWVkKQogICAgdHJ5OgogICAgICAgIGltcG9ydCB0b3JjaAogICAgICAgIHRvcmNoLm1hbnVhbF9zZWVkKHNlZWQpOyB0b3JjaC5jdWRhLm1hbnVhbF9zZWVkX2FsbChzZWVkKQogICAgZXhjZXB0IEltcG9ydEVycm9yOgogICAgICAgIHBhc3MKCmRlZiBnZXRfZGV2aWNlKCk6CiAgICBpbXBvcnQgdG9yY2gKICAgIHJldHVybiB0b3JjaC5kZXZpY2UoImN1ZGEiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikKCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLSBtZXRyaWNzCmRlZiBjbGZfbWV0cmljcyh5X3RydWUsIHlfcHJlZCwgbl9jbGFzc2VzKToKICAgICIiIk1pY3JvLUYxICg9PSBhY2N1cmFjeSBmb3Igc2luZ2xlLWxhYmVsKSBhbmQgbWFjcm8tRjEuIiIiCiAgICBmcm9tIHNrbGVhcm4ubWV0cmljcyBpbXBvcnQgZjFfc2NvcmUsIGFjY3VyYWN5X3Njb3JlCiAgICByZXR1cm4gewogICAgICAgICJtaWNyb19mMSI6IGZsb2F0KGYxX3Njb3JlKHlfdHJ1ZSwgeV9wcmVkLCBhdmVyYWdlPSJtaWNybyIsIHplcm9fZGl2aXNpb249MCkpLAogICAgICAgICJtYWNyb19mMSI6IGZsb2F0KGYxX3Njb3JlKHlfdHJ1ZSwgeV9wcmVkLCBhdmVyYWdlPSJtYWNybyIsIHplcm9fZGl2aXNpb249MCkpLAogICAgICAgICJhY2N1cmFjeSI6IGZsb2F0KGFjY3VyYWN5X3Njb3JlKHlfdHJ1ZSwgeV9wcmVkKSksCiAgICB9CgpkZWYgbXVsdGlsYWJlbF9tZXRyaWNzKHlfdHJ1ZSwgeV9wcm9iLCB0aHJlc2g9MC41KToKICAgICIiIkV4YW1wbGUtYmFzZWQgRjEgKEJMVVJCIGNvbnZlbnRpb24gZm9yIEhvQykgKyBtaWNyby9tYWNybyBGMS4iIiIKICAgIGZyb20gc2tsZWFybi5tZXRyaWNzIGltcG9ydCBmMV9zY29yZQogICAgeV9wcmVkID0gKHlfcHJvYiA+PSB0aHJlc2gpLmFzdHlwZShpbnQpCiAgICBpbnRlciA9ICh5X3ByZWQgKiB5X3RydWUpLnN1bSgxKQogICAgZGVub20gPSB5X3ByZWQuc3VtKDEpICsgeV90cnVlLnN1bSgxKQogICAgZXhfZjEgPSBucC53aGVyZShkZW5vbSA+IDAsIDIuMCAqIGludGVyIC8gbnAubWF4aW11bShkZW5vbSwgMWUtOSksIDEuMCkKICAgIHJldHVybiB7CiAgICAgICAgImV4YW1wbGVfZjEiOiBmbG9hdChleF9mMS5tZWFuKCkpLAogICAgICAgICJtaWNyb19mMSI6IGZsb2F0KGYxX3Njb3JlKHlfdHJ1ZSwgeV9wcmVkLCBhdmVyYWdlPSJtaWNybyIsIHplcm9fZGl2aXNpb249MCkpLAogICAgICAgICJtYWNyb19mMSI6IGZsb2F0KGYxX3Njb3JlKHlfdHJ1ZSwgeV9wcmVkLCBhdmVyYWdlPSJtYWNybyIsIHplcm9fZGl2aXNpb249MCkpLAogICAgfQoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIHBhcmFtIGNvdW50aW5nCmRlZiBjb3VudF9wYXJhbXMobW9kZWwpOgogICAgdG90YWwgPSBzdW0ocC5udW1lbCgpIGZvciBwIGluIG1vZGVsLnBhcmFtZXRlcnMoKSkKICAgIHRyYWluYWJsZSA9IHN1bShwLm51bWVsKCkgZm9yIHAgaW4gbW9kZWwucGFyYW1ldGVycygpIGlmIHAucmVxdWlyZXNfZ3JhZCkKICAgIHJldHVybiB0b3RhbCwgdHJhaW5hYmxlCgpkZWYgY291bnRfYWRhcHRlcl9wYXJhbXMobW9kZWwpOgogICAgIiIiVHJhaW5hYmxlIHBhcmFtcyBleGNsdWRpbmcgdGhlIHRhc2sgaGVhZCAoaGVhZCBpcyByZXF1aXJlZCBieSBldmVyeSBtZXRob2QsCiAgICBzbyBidWRnZXQgY29tcGFyaXNvbnMgYXJlIG1hZGUgb24gdGhlICphZGFwdGVyKiBwYXJhbWV0ZXJzIG9ubHkpLiIiIgogICAgbiA9IDAKICAgIGZvciBuYW1lLCBwIGluIG1vZGVsLm5hbWVkX3BhcmFtZXRlcnMoKToKICAgICAgICBpZiBwLnJlcXVpcmVzX2dyYWQgYW5kIG5vdCBuYW1lLnN0YXJ0c3dpdGgoImhlYWQuIik6CiAgICAgICAgICAgIG4gKz0gcC5udW1lbCgpCiAgICByZXR1cm4gbgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tIGlvCmRlZiBzYXZlX2pzb24ob2JqLCBwYXRoKToKICAgIG9zLm1ha2VkaXJzKG9zLnBhdGguZGlybmFtZShwYXRoKSwgZXhpc3Rfb2s9VHJ1ZSkKICAgIHdpdGggb3BlbihwYXRoLCAidyIsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAganNvbi5kdW1wKG9iaiwgZiwgaW5kZW50PTIsIGRlZmF1bHQ9c3RyKQoKZGVmIGxvYWRfanNvbihwYXRoKToKICAgIHdpdGggb3BlbihwYXRoLCBlbmNvZGluZz0idXRmLTgiKSBhcyBmOgogICAgICAgIHJldHVybiBqc29uLmxvYWQoZikKCmNsYXNzIFRpbWVyOgogICAgZGVmIF9fZW50ZXJfXyhzZWxmKTogc2VsZi50MCA9IHRpbWUucGVyZl9jb3VudGVyKCk7IHJldHVybiBzZWxmCiAgICBkZWYgX19leGl0X18oc2VsZiwgKmEpOiBzZWxmLmR0ID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIHNlbGYudDAK", "data.py": "IiIiRGF0YXNldCBsb2FkaW5nIGZvciBiaW9tZWRpY2FsIGNsYXNzaWZpY2F0aW9uIHRhc2tzICsgZ2VuZXJhbC1kb21haW4gcmVmZXJlbmNlIGNvcnB1cy4KClRhc2tzCi0tLS0tCmNoZW1wcm90IDogMTMtd2F5IGNoZW1pY2FsLXByb3RlaW4gcmVsYXRpb24gY2xhc3NpZmljYXRpb24sIHNlbnRlbmNlIGxldmVsLCBQdWJNZWQKICAgICAgICAgICBhYnN0cmFjdHMgd2l0aCBlbnRpdHkgbWVudGlvbnMgbWFya2VkIGJ5IDw8ID4+IGFuZCBbWyBdXS4gIENhbm9uaWNhbAogICAgICAgICAgIERBUFQvVEFQVCBhbmQgQkxVUkIgdGFzay4gIDQxNjkgLyAyNDI3IC8gMzQ2OS4KcmN0MjBrICAgOiA1LXdheSByaGV0b3JpY2FsLXJvbGUgY2xhc3NpZmljYXRpb24gb2Ygc2VudGVuY2VzIGluIFJDVCBhYnN0cmFjdHMKICAgICAgICAgICAoQkFDS0dST1VORCAvIE9CSkVDVElWRSAvIE1FVEhPRFMgLyBSRVNVTFRTIC8gQ09OQ0xVU0lPTlMpLgpob2MgICAgICA6IEhhbGxtYXJrcyBvZiBDYW5jZXIgLS0gMTAtbGFiZWwgbXVsdGktbGFiZWwgY2xhc3NpZmljYXRpb24gb2YgUHViTWVkCiAgICAgICAgICAgYWJzdHJhY3RzLiAgUmVjb25zdHJ1Y3RlZCBhdCBkb2N1bWVudCBsZXZlbCAodGhlIEJMVVJCIGZvcm11bGF0aW9uKSBieQogICAgICAgICAgIGdyb3VwaW5nIHRoZSBzZW50ZW5jZS1sZXZlbCByZWxlYXNlIG9uIFBNSUQgYW5kIHRha2luZyB0aGUgdW5pb24gb2YKICAgICAgICAgICBoYWxsbWFyayBsYWJlbHM7IHRoZSAibm8gaGFsbG1hcmsiIGNsYXNzIGlzIGRyb3BwZWQsIHNvIGFic3RyYWN0cyB3aXRoCiAgICAgICAgICAgbm8gaGFsbG1hcmsgY2FycnkgYW4gYWxsLXplcm8gdGFyZ2V0LgoKUmVmZXJlbmNlIGNvcnB1cwotLS0tLS0tLS0tLS0tLS0tCndpa2l0ZXh0LTEwMyAocmF3KSAtLSBhIGdlbmVyYWwtZG9tYWluIHByb3h5IGZvciB0aGUgcHJldHJhaW5pbmcgZGlzdHJpYnV0aW9uLCB1c2VkCmJ5IERSSUZUIHRvIGVzdGltYXRlIHRoZSBzdWJzcGFjZSB0aGUgYmFzZSBtb2RlbCBoYXMgYWxyZWFkeSBiZWVuIG9wdGltaXNlZCBmb3IuCiIiIgppbXBvcnQganNvbgppbXBvcnQgb3MKaW1wb3J0IHJhbmRvbQoKREFUQSA9IG9zLnBhdGguam9pbihvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpKSwgImRhdGEiKQoKCmRlZiBfcmVhZF9qc29ubChwYXRoKToKICAgIHJvd3MgPSBbXQogICAgd2l0aCBvcGVuKHBhdGgsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgZm9yIGxpbmUgaW4gZjoKICAgICAgICAgICAgbGluZSA9IGxpbmUuc3RyaXAoKQogICAgICAgICAgICBpZiBsaW5lOgogICAgICAgICAgICAgICAgcm93cy5hcHBlbmQoanNvbi5sb2FkcyhsaW5lKSkKICAgIHJldHVybiByb3dzCgoKZGVmIF9zdWJzYW1wbGUodGV4dHMsIGxhYmVscywgbiwgc2VlZD0wKToKICAgIGlmIG4gaXMgTm9uZSBvciBuID49IGxlbih0ZXh0cyk6CiAgICAgICAgcmV0dXJuIHRleHRzLCBsYWJlbHMKICAgIHJuZyA9IHJhbmRvbS5SYW5kb20oc2VlZCkKICAgIGlkeCA9IGxpc3QocmFuZ2UobGVuKHRleHRzKSkpCiAgICBybmcuc2h1ZmZsZShpZHgpCiAgICBpZHggPSBzb3J0ZWQoaWR4WzpuXSkKICAgIHJldHVybiBbdGV4dHNbaV0gZm9yIGkgaW4gaWR4XSwgW2xhYmVsc1tpXSBmb3IgaSBpbiBpZHhdCgoKZGVmIF9qc29ubF90YXNrKGZvbGRlciwgbWF4X3RyYWluPU5vbmUsIHNlZWQ9MCwgZXZhbF9jYXA9Tm9uZSwgbmFtZT0iIik6CiAgICByYXcsIGxhYmVsX3NldCA9IHt9LCBOb25lCiAgICBmb3Igc3BsaXQsIGZuIGluIFsoInRyYWluIiwgInRyYWluLmpzb25sIiksICgiZGV2IiwgImRldi5qc29ubCIpLCAoInRlc3QiLCAidGVzdC5qc29ubCIpXToKICAgICAgICByb3dzID0gX3JlYWRfanNvbmwob3MucGF0aC5qb2luKERBVEEsIGZvbGRlciwgZm4pKQogICAgICAgIHJhd1tzcGxpdF0gPSAoW3JbInRleHQiXSBmb3IgciBpbiByb3dzXSwgW3JbImxhYmVsIl0gZm9yIHIgaW4gcm93c10pCiAgICAgICAgaWYgbGFiZWxfc2V0IGlzIE5vbmU6CiAgICAgICAgICAgIGxhYmVsX3NldCA9IHNvcnRlZCh7clsibGFiZWwiXSBmb3IgciBpbiByb3dzfSkKICAgIGwyaSA9IHtsOiBpIGZvciBpLCBsIGluIGVudW1lcmF0ZShsYWJlbF9zZXQpfQogICAgb3V0ID0ge30KICAgIGZvciBzcGxpdCwgKHQsIGwpIGluIHJhdy5pdGVtcygpOgogICAgICAgIHkgPSBbbDJpW3hdIGZvciB4IGluIGxdCiAgICAgICAgaWYgc3BsaXQgPT0gInRyYWluIjoKICAgICAgICAgICAgdCwgeSA9IF9zdWJzYW1wbGUodCwgeSwgbWF4X3RyYWluLCBzZWVkKQogICAgICAgIGVsaWYgZXZhbF9jYXAgaXMgbm90IE5vbmU6CiAgICAgICAgICAgICMgQ2FwcGVkIHdpdGggYSBGSVhFRCBzZWVkIHNvIGV2ZXJ5IG1ldGhvZC9zZWVkIHNlZXMgdGhlIGlkZW50aWNhbAogICAgICAgICAgICAjIGV2YWx1YXRpb24gc3Vic2V0OyBjb21wYXJpc29ucyB0aGVyZWZvcmUgc3RheSBwYWlyZWQuCiAgICAgICAgICAgIHQsIHkgPSBfc3Vic2FtcGxlKHQsIHksIGV2YWxfY2FwLCAxMjM0NSkKICAgICAgICBvdXRbc3BsaXRdID0gKHQsIHkpCiAgICByZXR1cm4geyJzcGxpdHMiOiBvdXQsICJudW1fbGFiZWxzIjogbGVuKGxhYmVsX3NldCksICJsYWJlbHMiOiBsYWJlbF9zZXQsCiAgICAgICAgICAgICJtdWx0aWxhYmVsIjogRmFsc2UsICJtZXRyaWMiOiAibWljcm9fZjEiLCAibmFtZSI6IG5hbWV9CgoKZGVmIGxvYWRfY2hlbXByb3QobWF4X3RyYWluPU5vbmUsIHNlZWQ9MCk6CiAgICByZXR1cm4gX2pzb25sX3Rhc2soImNoZW1wcm90IiwgbWF4X3RyYWluLCBzZWVkLCBldmFsX2NhcD1Ob25lLCBuYW1lPSJjaGVtcHJvdCIpCgoKZGVmIGxvYWRfcmN0MjBrKG1heF90cmFpbj01MDAwLCBzZWVkPTApOgogICAgcmV0dXJuIF9qc29ubF90YXNrKCJyY3QyMGsiLCBtYXhfdHJhaW4sIHNlZWQsIGV2YWxfY2FwPTYwMDAsIG5hbWU9InJjdDIwayIpCgoKZGVmIGxvYWRfaG9jKG1heF90cmFpbj1Ob25lLCBzZWVkPTApOgogICAgaW1wb3J0IHBhbmRhcyBhcyBwZAogICAgaW1wb3J0IG51bXB5IGFzIG5wCiAgICBOT05FX0NMQVNTID0gNwogICAgZnJhbWVzID0ge30KICAgIGZvciBzcGxpdCwgZm4gaW4gWygidHJhaW4iLCAidHJhaW4ucGFycXVldCIpLCAoImRldiIsICJ2YWxpZGF0aW9uLnBhcnF1ZXQiKSwKICAgICAgICAgICAgICAgICAgICAgICgidGVzdCIsICJ0ZXN0LnBhcnF1ZXQiKV06CiAgICAgICAgZGYgPSBwZC5yZWFkX3BhcnF1ZXQob3MucGF0aC5qb2luKERBVEEsICJob2MiLCBmbikpCiAgICAgICAgZGZbInBtaWQiXSA9IGRmWyJkb2N1bWVudF9pZCJdLnN0ci5zcGxpdCgiXyIpLnN0clswXQogICAgICAgIGRmWyJzaWR4Il0gPSBkZlsiZG9jdW1lbnRfaWQiXS5zdHIuc3BsaXQoIl8iKS5zdHJbMV0uYXN0eXBlKGludCkKICAgICAgICBnID0gZGYuc29ydF92YWx1ZXMoWyJwbWlkIiwgInNpZHgiXSkuZ3JvdXBieSgicG1pZCIpCiAgICAgICAgdGV4dHMgPSBnWyJ0ZXh0Il0uYXBwbHkobGFtYmRhIHM6ICIgIi5qb2luKHMpKQogICAgICAgIGxhYnMgPSBnWyJsYWJlbCJdLmFwcGx5KAogICAgICAgICAgICBsYW1iZGEgczogc29ydGVkKHtpbnQoeCkgZm9yIGwgaW4gcyBmb3IgeCBpbiBsIGlmIGludCh4KSAhPSBOT05FX0NMQVNTfSkpCiAgICAgICAgZnJhbWVzW3NwbGl0XSA9ICh0ZXh0cy50b2xpc3QoKSwgbGFicy50b2xpc3QoKSkKCiAgICBwcmVzZW50ID0gc29ydGVkKHt4IGZvciBfLCBsYWJzIGluIGZyYW1lcy52YWx1ZXMoKSBmb3IgbCBpbiBsYWJzIGZvciB4IGluIGx9KQogICAgbDJpID0ge2M6IGkgZm9yIGksIGMgaW4gZW51bWVyYXRlKHByZXNlbnQpfQogICAgb3V0ID0ge30KICAgIGZvciBzcGxpdCwgKHQsIGxhYnMpIGluIGZyYW1lcy5pdGVtcygpOgogICAgICAgIHkgPSBucC56ZXJvcygobGVuKGxhYnMpLCBsZW4ocHJlc2VudCkpLCBkdHlwZT0iZmxvYXQzMiIpCiAgICAgICAgZm9yIGksIGwgaW4gZW51bWVyYXRlKGxhYnMpOgogICAgICAgICAgICBmb3IgYyBpbiBsOgogICAgICAgICAgICAgICAgeVtpLCBsMmlbY11dID0gMS4wCiAgICAgICAgeSA9IFtyb3cgZm9yIHJvdyBpbiB5XQogICAgICAgIGlmIHNwbGl0ID09ICJ0cmFpbiI6CiAgICAgICAgICAgIHQsIHkgPSBfc3Vic2FtcGxlKHQsIHksIG1heF90cmFpbiwgc2VlZCkKICAgICAgICBvdXRbc3BsaXRdID0gKHQsIHkpCiAgICByZXR1cm4geyJzcGxpdHMiOiBvdXQsICJudW1fbGFiZWxzIjogbGVuKHByZXNlbnQpLCAibGFiZWxzIjogcHJlc2VudCwKICAgICAgICAgICAgIm11bHRpbGFiZWwiOiBUcnVlLCAibWV0cmljIjogImV4YW1wbGVfZjEiLCAibmFtZSI6ICJob2MifQoKCmRlZiBsb2FkX3Rhc2sobmFtZSwgbWF4X3RyYWluPU5vbmUsIHNlZWQ9MCk6CiAgICBpZiBuYW1lID09ICJjaGVtcHJvdCI6CiAgICAgICAgcmV0dXJuIGxvYWRfY2hlbXByb3QobWF4X3RyYWluLCBzZWVkKQogICAgaWYgbmFtZSA9PSAicmN0MjBrIjoKICAgICAgICByZXR1cm4gbG9hZF9yY3QyMGsobWF4X3RyYWluIGlmIG1heF90cmFpbiBpcyBub3QgTm9uZSBlbHNlIDUwMDAsIHNlZWQpCiAgICBpZiBuYW1lID09ICJob2MiOgogICAgICAgIHJldHVybiBsb2FkX2hvYyhtYXhfdHJhaW4sIHNlZWQpCiAgICByYWlzZSBWYWx1ZUVycm9yKCJ1bmtub3duIHRhc2sgIiArIG5hbWUpCgoKVEFTS19NQVhMRU4gPSB7ImNoZW1wcm90IjogMTI4LCAicmN0MjBrIjogOTYsICJob2MiOiA1MTJ9CgoKZGVmIGxvYWRfcmVmZXJlbmNlX2NvcnB1cyhuX2RvY3M9MjAwMCwgbWluX2NoYXJzPTIwMCwgc2VlZD0wKToKICAgICIiIkdlbmVyYWwtZG9tYWluIHJlZmVyZW5jZSB0ZXh0ICh3aWtpdGV4dC0xMDMgcmF3KS4iIiIKICAgIGltcG9ydCBwYW5kYXMgYXMgcGQKICAgIGRmcyA9IFtdCiAgICBmb3IgZm4gaW4gKCJ3aWtpdGV4dF92YWwucGFycXVldCIsICJ3aWtpdGV4dF90ZXN0LnBhcnF1ZXQiKToKICAgICAgICBwID0gb3MucGF0aC5qb2luKERBVEEsICJyZWZlcmVuY2UiLCBmbikKICAgICAgICBpZiBvcy5wYXRoLmV4aXN0cyhwKToKICAgICAgICAgICAgZGZzLmFwcGVuZChwZC5yZWFkX3BhcnF1ZXQocCkpCiAgICBkZiA9IHBkLmNvbmNhdChkZnMsIGlnbm9yZV9pbmRleD1UcnVlKQogICAgdGV4dHMgPSBbdC5zdHJpcCgpIGZvciB0IGluIGRmWyJ0ZXh0Il0udG9saXN0KCkgaWYgaXNpbnN0YW5jZSh0LCBzdHIpXQogICAgdGV4dHMgPSBbdCBmb3IgdCBpbiB0ZXh0cyBpZiBsZW4odCkgPj0gbWluX2NoYXJzIGFuZCBub3QgdC5zdGFydHN3aXRoKCI9IildCiAgICBybmcgPSByYW5kb20uUmFuZG9tKHNlZWQpCiAgICBybmcuc2h1ZmZsZSh0ZXh0cykKICAgIHJldHVybiB0ZXh0c1s6bl9kb2NzXQo=", "drift.py": "IiIiRFJJRlQ6IERvbWFpbi1SZXNpZHVhbCBJbmZvcm1lZCBGaW5lLVR1bmluZy4KClRyYWluaW5nLWZyZWUsIGdyYWRpZW50LWZyZWUsIGxhYmVsLWZyZWUgcHJvZmlsaW5nIHRoYXQgZGVjaWRlcyAoYSkgaG93IG11Y2ggTG9SQQpyYW5rIGVhY2ggbGluZWFyIG1vZHVsZSByZWNlaXZlcyBhbmQgKGIpIHdoaWNoIHN1YnNwYWNlIGl0cyBhZGFwdGVyIGlzIGluaXRpYWxpc2VkIGluLgoKQ29yZSBpZGVhCi0tLS0tLS0tLQpFeGlzdGluZyBhY3RpdmF0aW9uLWdlb21ldHJ5IG1ldGhvZHMgKEVWQSwgQ29yREEsIEFJUkEsIFRMb1JBLCBSU0xvUkEpIGNoYXJhY3RlcmlzZQp0aGUgKnRhcmdldCogYWN0aXZhdGlvbiBkaXN0cmlidXRpb24gaW4gaXNvbGF0aW9uLiAgRm9yIGRvbWFpbiBhZGFwdGF0aW9uIHRoZSB1c2VmdWwKcXVlc3Rpb24gaXMgZGlmZmVyZW50OiB3aGljaCBkaXJlY3Rpb25zIG9mIHRoZSB0YXJnZXQtZG9tYWluIHJlcHJlc2VudGF0aW9uIGFyZSBvbmVzCnRoZSBwcmV0cmFpbmVkIG1vZGVsIGhhcyBuZXZlciBoYWQgdG8gbW9kZWw/ICBXZSBhbnN3ZXIgaXQgYnkgY29udHJhc3RpbmcgdGhlIHRhcmdldApjb3ZhcmlhbmNlIGFnYWluc3QgdGhhdCBvZiBhIGdlbmVyYWwtZG9tYWluIHJlZmVyZW5jZSBjb3JwdXM6CgogICAgU2lnbWFfRyA9IENvdl9HW3hdICAgICAgICAgICAgKHJlZmVyZW5jZSAvIHByZXRyYWluaW5nIHByb3h5KQogICAgU2lnbWFfRCA9IENvdl9EW3hdICAgICAgICAgICAgKHRhcmdldCBkb21haW4pCiAgICBQX0cgICAgID0gVV9rIFVfa15UICAgICAgICAgICAodG9wLWsgZWlnZW5zcGFjZSBvZiBTaWdtYV9HIGNhcHR1cmluZyBlbmVyZ3kgdGF1KQogICAgU2lnbWF+ICA9IChJIC0gUF9HKSBTaWdtYV9EIChJIC0gUF9HKSAgICAgIDwtLSB0aGUgKmRyaWZ0KiAocmVzaWR1YWwpIGNvdmFyaWFuY2UKClJhbmsgaXMgYWxsb2NhdGVkIGJ5IGdyZWVkeSBtYXJnaW5hbCBjb3ZlcmFnZSBvZiB0aGUgZHJpZnQgc3BlY3RydW0gdW5kZXIgYSBnbG9iYWwKcGFyYW1ldGVyIGJ1ZGdldCAocHJvdmFibHkgb3B0aW1hbCwgc2VlIGFsbG9jYXRlX3JhbmtzKSwgYW5kIGFkYXB0ZXJzIGFyZSBpbml0aWFsaXNlZAp3aXRoIHRoZSBsZWFkaW5nIGRyaWZ0IGVpZ2VudmVjdG9ycy4KCnRhdSBpbnRlcnBvbGF0ZXMgdGhlIG1ldGhvZCBmYW1pbHk6IHRhdSAtPiAwIGdpdmVzIGsgPSAwLCBQX0cgPSAwIGFuZCBTaWdtYX4gPSBTaWdtYV9ELAppLmUuIHBsYWluIGluLWRvbWFpbiBhY3RpdmF0aW9uIFBDQSAoRVZBKS4gIHRhdSA+IDAgZGVmbGF0ZXMgdGhlIGRpcmVjdGlvbnMgdGhlIGJhc2UKbW9kZWwgYWxyZWFkeSBjb3ZlcnMuCiIiIgppbXBvcnQgZ2MKaW1wb3J0IGhlYXBxCmltcG9ydCBudW1weSBhcyBucAppbXBvcnQgdG9yY2gKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgbW9kdWxlIGRpc2NvdmVyeQojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCmRlZiBmaW5kX3RhcmdldF9tb2R1bGVzKG1vZGVsLCBpbmNsdWRlX2Zmbj1UcnVlLCBpbmNsdWRlX2F0dG49VHJ1ZSk6CiAgICAiIiJSZXR1cm4ge25hbWU6IG5uLkxpbmVhcn0gZm9yIHRoZSBhZGFwdGFibGUgbGluZWFyIG1vZHVsZXMgb2YgYW4gZW5jb2Rlci4iIiIKICAgIGltcG9ydCB0b3JjaC5ubiBhcyBubgogICAgb3V0ID0ge30KICAgIGZvciBuYW1lLCBtb2QgaW4gbW9kZWwubmFtZWRfbW9kdWxlcygpOgogICAgICAgIGlmIG5vdCBpc2luc3RhbmNlKG1vZCwgbm4uTGluZWFyKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBsb3cgPSBuYW1lLmxvd2VyKCkKICAgICAgICBpZiAoImVtYmVkZGluZ3MiIGluIGxvdyBvciBsb3cuc3RhcnRzd2l0aCgiaGVhZCIpIG9yICJjbGFzc2lmaWVyIiBpbiBsb3cKICAgICAgICAgICAgICAgIG9yICJwb29sZXIiIGluIGxvdyk6CiAgICAgICAgICAgIGNvbnRpbnVlCiAgICAgICAgaXNfYXR0biA9IGFueShrIGluIGxvdyBmb3IgayBpbiAoInF1ZXJ5IiwgImtleSIsICJ2YWx1ZSIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgImF0dGVudGlvbi5vdXRwdXQuZGVuc2UiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJxX3Byb2oiLCAia19wcm9qIiwgInZfcHJvaiIsICJvdXRfcHJvaiIsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIm9fcHJvaiIpKSAgICAgICAgICAjIExsYW1hLXN0eWxlIGRlY29kZXJzCiAgICAgICAgaXNfZmZuID0gKCgiaW50ZXJtZWRpYXRlLmRlbnNlIiBpbiBsb3cpCiAgICAgICAgICAgICAgICAgIG9yIChsb3cuZW5kc3dpdGgoIm91dHB1dC5kZW5zZSIpIGFuZCAiYXR0ZW50aW9uIiBub3QgaW4gbG93KQogICAgICAgICAgICAgICAgICBvciAiZmMxIiBpbiBsb3cgb3IgImZjMiIgaW4gbG93CiAgICAgICAgICAgICAgICAgIG9yICJnYXRlX3Byb2oiIGluIGxvdyBvciAidXBfcHJvaiIgaW4gbG93IG9yICJkb3duX3Byb2oiIGluIGxvdykKICAgICAgICBpZiAoaXNfYXR0biBhbmQgaW5jbHVkZV9hdHRuKSBvciAoaXNfZmZuIGFuZCBpbmNsdWRlX2Zmbik6CiAgICAgICAgICAgIG91dFtuYW1lXSA9IG1vZAogICAgcmV0dXJuIG91dAoKCmRlZiBlbnN1cmVfcGFkZGluZyh0b2tlbml6ZXIsIG1vZGVsPU5vbmUpOgogICAgIiIiRGVjb2RlciB0b2tlbml6ZXJzIG9mdGVuIHNoaXAgd2l0aG91dCBhIHBhZGRpbmcgdG9rZW4sIGFuZCBhIGRlY29kZXIncwogICAgc2VxdWVuY2UtY2xhc3NpZmljYXRpb24gaGVhZCBuZWVkcyBvbmUgdG8gZmluZCBlYWNoIHNlcXVlbmNlJ3MgbGFzdCB0b2tlbi4KICAgIFJldXNlIEVPUyBhbmQgcGFkIG9uIHRoZSByaWdodCwgYXMgdGhlIHRyYWluaW5nIGNvbGxhdGUgZG9lcy4iIiIKICAgIGlmIHRva2VuaXplci5wYWRfdG9rZW4gaXMgTm9uZToKICAgICAgICB0b2tlbml6ZXIucGFkX3Rva2VuID0gdG9rZW5pemVyLmVvc190b2tlbgogICAgdG9rZW5pemVyLnBhZGRpbmdfc2lkZSA9ICJyaWdodCIKICAgIGlmIG1vZGVsIGlzIG5vdCBOb25lIGFuZCBnZXRhdHRyKG1vZGVsLmNvbmZpZywgInBhZF90b2tlbl9pZCIsIE5vbmUpIGlzIE5vbmU6CiAgICAgICAgbW9kZWwuY29uZmlnLnBhZF90b2tlbl9pZCA9IHRva2VuaXplci5wYWRfdG9rZW5faWQKICAgIHJldHVybiB0b2tlbml6ZXIKCgpkZWYgbW9kdWxlX2Nvc3QobW9kKToKICAgICIiIlBhcmFtZXRlcnMgY29uc3VtZWQgcGVyIHVuaXQgb2YgcmFuazogQSBpcyAociB4IGRfaW4pLCBCIGlzIChkX291dCB4IHIpLiIiIgogICAgcmV0dXJuIG1vZC5pbl9mZWF0dXJlcyArIG1vZC5vdXRfZmVhdHVyZXMKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgc3RyZWFtaW5nIHNlY29uZC1tb21lbnQgYWNjdW11bGF0aW9uCiMgLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0KY2xhc3MgX0Nvdkhvb2s6CiAgICAiIiJBY2N1bXVsYXRlcyBmaXJzdCBhbmQgc2Vjb25kIG1vbWVudHMgb3ZlciBub24tcGFkZGluZyB0b2tlbiBwb3NpdGlvbnMgb2YgYQogICAgbW9kdWxlIGlucHV0LgoKICAgIElucHV0cyBhcmUgc2hpZnRlZCBieSB0aGUgZmlyc3QgYmF0Y2gncyBtZWFuIGJlZm9yZSBhY2N1bXVsYXRpb24uIFRyYW5zZm9ybWVyCiAgICBhY3RpdmF0aW9ucyBoYXZlIGEgbGFyZ2UgbWVhbiAoYW5kIGEgZmV3IG1hc3NpdmUgb3V0bGllciBkaW1lbnNpb25zKSwgc28KICAgIGZvcm1pbmcgRVt4eF5UXSAtIG11IG11XlQgZGlyZWN0bHkgaW4gZmxvYXQzMiB3b3VsZCBjYW5jZWwgY2F0YXN0cm9waGljYWxseTsKICAgIHdpdGggdGhlIHNoaWZ0LCB0aGUgZmluYWwgbWVhbiBjb3JyZWN0aW9uIGlzIHNtYWxsLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGQsIGRldmljZSwgZHR5cGU9dG9yY2guZmxvYXQzMik6CiAgICAgICAgc2VsZi5hY2MgPSB0b3JjaC56ZXJvcyhkLCBkLCBkZXZpY2U9ZGV2aWNlLCBkdHlwZT1kdHlwZSkKICAgICAgICBzZWxmLnN1bSA9IHRvcmNoLnplcm9zKGQsIGRldmljZT1kZXZpY2UsIGR0eXBlPWR0eXBlKQogICAgICAgIHNlbGYuc2hpZnQgPSBOb25lCiAgICAgICAgc2VsZi5uID0gMAogICAgICAgIHNlbGYubWFzayA9IE5vbmUKCiAgICBkZWYgX19jYWxsX18oc2VsZiwgbW9kdWxlLCBpbnB1dHMsIG91dHB1dCk6CiAgICAgICAgeCA9IGlucHV0c1swXQogICAgICAgIGlmIHguZGltKCkgPT0gMzogICAgICAgICAgICAgICAgICAgICAgICMgKEIsIFQsIGQpCiAgICAgICAgICAgIGlmIHNlbGYubWFzayBpcyBub3QgTm9uZToKICAgICAgICAgICAgICAgIG0gPSBzZWxmLm1hc2sucmVzaGFwZSgtMSkuYm9vbCgpCiAgICAgICAgICAgICAgICB4ID0geC5yZXNoYXBlKC0xLCB4LnNoYXBlWy0xXSlbbV0KICAgICAgICAgICAgZWxzZToKICAgICAgICAgICAgICAgIHggPSB4LnJlc2hhcGUoLTEsIHguc2hhcGVbLTFdKQogICAgICAgIHggPSB4LnRvKHNlbGYuYWNjLmR0eXBlKQogICAgICAgIGlmIHNlbGYuc2hpZnQgaXMgTm9uZToKICAgICAgICAgICAgc2VsZi5zaGlmdCA9IHgubWVhbigwKQogICAgICAgIHggPSB4IC0gc2VsZi5zaGlmdAogICAgICAgIHNlbGYuYWNjICs9IHguVCBAIHgKICAgICAgICBzZWxmLnN1bSArPSB4LnN1bSgwKQogICAgICAgIHNlbGYubiArPSB4LnNoYXBlWzBdCgogICAgZGVmIG1vbWVudHMoc2VsZiwgY2VudGVyPVRydWUpOgogICAgICAgICIiIkNvdmFyaWFuY2UgKGNlbnRlcj1UcnVlKSBvciByYXcgc2Vjb25kIG1vbWVudCwgaW4gZmxvYXQ2NCBvbiBDUFUuIiIiCiAgICAgICAgbiA9IG1heChzZWxmLm4sIDEpCiAgICAgICAgYWNjID0gc2VsZi5hY2MuZG91YmxlKCkuY3B1KCkgLyBuCiAgICAgICAgbXMgPSBzZWxmLnN1bS5kb3VibGUoKS5jcHUoKSAvIG4gICAgICAgICAgICAjIG1lYW4gb2YgdGhlIHNoaWZ0ZWQgaW5wdXRzCiAgICAgICAgY292ID0gYWNjIC0gdG9yY2gub3V0ZXIobXMsIG1zKQogICAgICAgIGlmIGNlbnRlcjoKICAgICAgICAgICAgcmV0dXJuIGNvdgogICAgICAgIG11ID0gbXMgKyBzZWxmLnNoaWZ0LmRvdWJsZSgpLmNwdSgpCiAgICAgICAgcmV0dXJuIGNvdiArIHRvcmNoLm91dGVyKG11LCBtdSkKCgpkZWYgX2JhdGNoZWQoc2VxLCBicyk6CiAgICBmb3IgaSBpbiByYW5nZSgwLCBsZW4oc2VxKSwgYnMpOgogICAgICAgIHlpZWxkIHNlcVtpOmkgKyBic10KCgpAdG9yY2gubm9fZ3JhZCgpCmRlZiBjb2xsZWN0X2NvdmFyaWFuY2VzKG1vZGVsLCB0b2tlbml6ZXIsIHRleHRzLCBtb2R1bGVzLCBkZXZpY2UsIG1heF9sZW49MTI4LAogICAgICAgICAgICAgICAgICAgICAgICBiYXRjaF9zaXplPTE2LCBkdHlwZT10b3JjaC5mbG9hdDMyLCBjZW50ZXI9VHJ1ZSk6CiAgICAiIiJGb3J3YXJkLW9ubHkgcGFzczsgcmV0dXJucyB7bmFtZTogKFNpZ21hLCBuX3Rva2Vucyl9IHdpdGggU2lnbWEgb24gQ1BVIGZsb2F0MzIuCgogICAgU2lnbWEgaXMgdGhlIGNvdmFyaWFuY2Ugb2YgdGhlIG1vZHVsZSBpbnB1dCAoY2VudGVyPVRydWUsIHRoZSBkZWZhdWx0LCBtYXRjaGluZwogICAgRVZBJ3MgcmVmZXJlbmNlIGltcGxlbWVudGF0aW9uLCB3aG9zZSBpbmNyZW1lbnRhbCBQQ0EgYWx3YXlzIGNlbnRyZXMpIG9yIHRoZQogICAgcmF3IHNlY29uZCBtb21lbnQgKGNlbnRlcj1GYWxzZSkuCgogICAgS2VwdCBpbiBmbG9hdDMyIGRlbGliZXJhdGVseTogYSBmdWxsIHNldCBvZiBzZWNvbmQgbW9tZW50cyBmb3IgYSAxMjVNIGVuY29kZXIKICAgIGlzIH4wLjYgR0IgaW4gZmxvYXQzMiBhbmQgfjEuMiBHQiBpbiBmbG9hdDY0LCBhbmQgaG9sZGluZyB0d28gb2YgdGhvc2UKICAgIChyZWZlcmVuY2UgYW5kIHRhcmdldCkgaW4gZmxvYXQ2NCBpcyBlbm91Z2ggdG8gcHVzaCBhbiA4IEdCIG1hY2hpbmUgaW50bwogICAgc3dhcHBpbmcsIHdoaWNoIGNvc3RzIGZhciBtb3JlIHRoYW4gdGhlIHByZWNpc2lvbiBpcyB3b3J0aC4gVGhlIHNwZWN0cmFsCiAgICBzdGFnZSBwcm9tb3RlcyBvbmUgbW9kdWxlIGF0IGEgdGltZSB0byBmbG9hdDY0LgogICAgIiIiCiAgICBob29rcywgaGFuZGxlcyA9IHt9LCBbXQogICAgZm9yIG5hbWUsIG1vZCBpbiBtb2R1bGVzLml0ZW1zKCk6CiAgICAgICAgaCA9IF9Db3ZIb29rKG1vZC5pbl9mZWF0dXJlcywgZGV2aWNlLCBkdHlwZSkKICAgICAgICBob29rc1tuYW1lXSA9IGgKICAgICAgICBoYW5kbGVzLmFwcGVuZChtb2QucmVnaXN0ZXJfZm9yd2FyZF9ob29rKGgpKQoKICAgIG1vZGVsLmV2YWwoKQogICAga2VlcCA9ICgiaW5wdXRfaWRzIiwgImF0dGVudGlvbl9tYXNrIiwgInRva2VuX3R5cGVfaWRzIikKICAgIGZvciBiYXRjaCBpbiBfYmF0Y2hlZCh0ZXh0cywgYmF0Y2hfc2l6ZSk6CiAgICAgICAgZW5jID0gdG9rZW5pemVyKGxpc3QoYmF0Y2gpLCB0cnVuY2F0aW9uPVRydWUsIG1heF9sZW5ndGg9bWF4X2xlbiwKICAgICAgICAgICAgICAgICAgICAgICAgcGFkZGluZz1UcnVlLCByZXR1cm5fdGVuc29ycz0icHQiKS50byhkZXZpY2UpCiAgICAgICAgYW0gPSBlbmNbImF0dGVudGlvbl9tYXNrIl0KICAgICAgICBmb3IgaCBpbiBob29rcy52YWx1ZXMoKToKICAgICAgICAgICAgaC5tYXNrID0gYW0KICAgICAgICBtb2RlbCgqKntrOiB2IGZvciBrLCB2IGluIGVuYy5pdGVtcygpIGlmIGsgaW4ga2VlcH0pCgogICAgZm9yIGggaW4gaGFuZGxlczoKICAgICAgICBoLnJlbW92ZSgpCiAgICBvdXQgPSB7fQogICAgZm9yIG5hbWUsIGggaW4gaG9va3MuaXRlbXMoKToKICAgICAgICBvdXRbbmFtZV0gPSAoaC5tb21lbnRzKGNlbnRlcikuZmxvYXQoKSwgaC5uKQogICAgICAgIGguYWNjID0gaC5zdW0gPSBOb25lCiAgICBkZWwgaG9va3MKICAgIGdjLmNvbGxlY3QoKQogICAgaWYgZGV2aWNlLnR5cGUgPT0gImN1ZGEiOgogICAgICAgIHRvcmNoLmN1ZGEuZW1wdHlfY2FjaGUoKQogICAgcmV0dXJuIG91dAoKCmRlZiBjaHVua19tb2R1bGVzKG1vZHVsZXMsIGJ1ZGdldF9ieXRlcz00MDBfMDAwXzAwMCwgbl9jb3Jwb3JhPTIsIGJ5dGVzX3Blcj04KToKICAgICIiIlNwbGl0IG1vZHVsZXMgaW50byBncm91cHMgd2hvc2UgY292YXJpYW5jZSBtYXRyaWNlcyBmaXQgaW4gYnVkZ2V0X2J5dGVzLiIiIgogICAgZ3JvdXBzLCBjdXIsIGN1cl9iID0gW10sIHt9LCAwCiAgICBmb3IgbmFtZSwgbW9kIGluIG1vZHVsZXMuaXRlbXMoKToKICAgICAgICBiID0gbW9kLmluX2ZlYXR1cmVzICoqIDIgKiBieXRlc19wZXIgKiBuX2NvcnBvcmEKICAgICAgICBpZiBjdXIgYW5kIGN1cl9iICsgYiA+IGJ1ZGdldF9ieXRlczoKICAgICAgICAgICAgZ3JvdXBzLmFwcGVuZChjdXIpCiAgICAgICAgICAgIGN1ciwgY3VyX2IgPSB7fSwgMAogICAgICAgIGN1cltuYW1lXSA9IG1vZAogICAgICAgIGN1cl9iICs9IGIKICAgIGlmIGN1cjoKICAgICAgICBncm91cHMuYXBwZW5kKGN1cikKICAgIHJldHVybiBncm91cHMKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgc3Vic3BhY2UgY29udHJhc3QKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQpkZWYgcmVmZXJlbmNlX2VpZ2goc2lnbWFfZyk6CiAgICAiIiJEZXNjZW5kaW5nIGVpZ2VuZGVjb21wb3NpdGlvbiBvZiB0aGUgcmVmZXJlbmNlIHNlY29uZCBtb21lbnQuCgogICAgQ29tcHV0ZWQgb25jZSBwZXIgbW9kdWxlIGFuZCByZXVzZWQgYWNyb3NzIGV2ZXJ5IHRhdSAtLSB0aGUgZGVjb21wb3NpdGlvbiBkb2VzCiAgICBub3QgZGVwZW5kIG9uIHRhdSwgb25seSB0aGUgdHJ1bmNhdGlvbiBwb2ludCBkb2VzLgogICAgIiIiCiAgICBldmFscywgZXZlY3MgPSB0b3JjaC5saW5hbGcuZWlnaChzaWdtYV9nKSAgICAgICAgICAjIGFzY2VuZGluZwogICAgcmV0dXJuIHRvcmNoLmZsaXAoZXZhbHMsIFswXSkuY2xhbXBfbWluKDApLCB0b3JjaC5mbGlwKGV2ZWNzLCBbMV0pCgoKZGVmIHN1YnNwYWNlX2Zyb21fZWlnaChldmFsc19nLCBldmVjc19nLCB0YXU9MC45NSwga19tYXg9Tm9uZSk6CiAgICAiIiJUcnVuY2F0ZSBhIHByZWNvbXB1dGVkIHJlZmVyZW5jZSBlaWdlbmJhc2lzIGF0IGVuZXJneSBmcmFjdGlvbiB0YXUuIiIiCiAgICBkID0gZXZlY3NfZy5zaGFwZVswXQogICAgaWYgdGF1IDw9IDA6CiAgICAgICAgcmV0dXJuIGV2ZWNzX2dbOiwgOjBdLCAwCiAgICB0b3QgPSBldmFsc19nLnN1bSgpCiAgICBpZiB0b3QgPD0gMDoKICAgICAgICByZXR1cm4gZXZlY3NfZ1s6LCA6MF0sIDAKICAgIGNzdW0gPSB0b3JjaC5jdW1zdW0oZXZhbHNfZywgMCkgLyB0b3QKICAgIGsgPSBpbnQodG9yY2guc2VhcmNoc29ydGVkKGNzdW0sIHRvcmNoLnRlbnNvcih0YXUsIGR0eXBlPWNzdW0uZHR5cGUpKS5pdGVtKCkpICsgMQogICAgayA9IG1pbihrLCBkIC0gMSkKICAgIGlmIGtfbWF4IGlzIG5vdCBOb25lOgogICAgICAgIGsgPSBtaW4oaywga19tYXgpCiAgICByZXR1cm4gZXZlY3NfZ1s6LCA6a10uY29udGlndW91cygpLCBrCgoKZGVmIHJlZmVyZW5jZV9zdWJzcGFjZShzaWdtYV9nLCB0YXU9MC45NSwga19tYXg9Tm9uZSk6CiAgICAiIiJDb252ZW5pZW5jZSB3cmFwcGVyOiBlaWdlbmRlY29tcG9zZSBhbmQgdHJ1bmNhdGUgaW4gb25lIGNhbGwuIiIiCiAgICBpZiB0YXUgPD0gMDoKICAgICAgICByZXR1cm4gdG9yY2guemVyb3Moc2lnbWFfZy5zaGFwZVswXSwgMCwgZHR5cGU9c2lnbWFfZy5kdHlwZSksIDAKICAgIGV2LCBldmVjID0gcmVmZXJlbmNlX2VpZ2goc2lnbWFfZykKICAgIHJldHVybiBzdWJzcGFjZV9mcm9tX2VpZ2goZXYsIGV2ZWMsIHRhdSwga19tYXgpCgoKZGVmIGRyaWZ0X3NwZWN0cnVtKHNpZ21hX2QsIHZfY29tcCwgcl9rZWVwPTY0LCBkZXZpY2U9Tm9uZSk6CiAgICAiIiJTcGVjdHJ1bSBvZiBTaWdtYX4gPSAoSS1QKSBTaWdtYV9EIChJLVApLCBjb21wdXRlZCBpbiB0aGUgY29tcGxlbWVudCBiYXNpcy4KCiAgICBgdl9jb21wYCBpcyBhIChkLCBkLWspIG9ydGhvbm9ybWFsIGJhc2lzIG9mIHRoZSBvcnRob2dvbmFsIGNvbXBsZW1lbnQgb2YgdGhlCiAgICByZWZlcmVuY2Ugc3Vic3BhY2UsIGkuZS4gdGhlICp0cmFpbGluZyogcmVmZXJlbmNlIGVpZ2VudmVjdG9ycywgc28gdGhhdAogICAgSSAtIFAgPSBWIFZeVC4gIFRoZW4gU2lnbWF+ID0gViBNIFZeVCB3aXRoIE0gPSBWXlQgU2lnbWFfRCBWLCBhbmQgdGhlIHR3bwogICAgc2hhcmUgZXZlcnkgbm9uemVybyBlaWdlbnZhbHVlIHdoaWxlIHRoZSBlaWdlbnZlY3RvcnMgYXJlIHJlbGF0ZWQgYnkgVi4KCiAgICBXb3JraW5nIHdpdGggTSBpbnN0ZWFkIG9mIFNpZ21hfiBpcyBib3RoIGNoZWFwZXIgYW5kIGJldHRlciBjb25kaXRpb25lZDogdGhlCiAgICBlaWdlbmRlY29tcG9zaXRpb24gc2hyaW5rcyBmcm9tIGReMyB0byAoZC1rKV4zIC0tIGF0IHRhdSA9IDAuOTUgdGhlIHJlZmVyZW5jZQogICAgc3Vic3BhY2UgdHlwaWNhbGx5IGFic29yYnMgbW9zdCBvZiB0aGUgc3BhY2UsIHNvIHRoaXMgaXMgYSBsYXJnZSBzYXZpbmcgLS0KICAgIGFuZCBmb3JtaW5nIE0gYXZvaWRzIHRoZSBjYXRhc3Ryb3BoaWMgY2FuY2VsbGF0aW9uIG9mIHN1YnRyYWN0aW5nIHR3byBuZWFybHkKICAgIGVxdWFsIGQgeCBkIG1hdHJpY2VzLgoKICAgIFBhc3MgYHZfY29tcGAgd2l0aCB6ZXJvIGNvbHVtbnMgdG8gbWVhbiAibm8gZGVmbGF0aW9uIiAodGF1ID0gMCksIGluIHdoaWNoCiAgICBjYXNlIHRoZSBwbGFpbiBzcGVjdHJ1bSBvZiBTaWdtYV9EIGlzIHJldHVybmVkLgoKICAgIFJldHVybnMgKGVpZ3ZhbHNfZGVzYywgdG9wLXJfa2VlcCBlaWd2ZWNzIGluIHRoZSBvcmlnaW5hbCBzcGFjZSwKICAgICAgICAgICAgIHRyYWNlKFNpZ21hX0QpLCB0cmFjZShTaWdtYX4pKS4KICAgICIiIgogICAgdHJhY2VfZCA9IGZsb2F0KHRvcmNoLmRpYWdvbmFsKHNpZ21hX2QpLnN1bSgpKQogICAgZCA9IHNpZ21hX2Quc2hhcGVbMF0KCiAgICBpZiB2X2NvbXAgaXMgTm9uZTogICAgICAgICAgICAgICAgICAgICAgICMgZXhwbGljaXQgIm5vIGRlZmxhdGlvbiIKICAgICAgICBtLCBiYWNrID0gc2lnbWFfZCwgTm9uZQogICAgZWxpZiB2X2NvbXAuc2hhcGVbMV0gPT0gMDogICAgICAgICAgICAgICAjIHJlZmVyZW5jZSBzdWJzcGFjZSBmaWxscyB0aGUgc3BhY2UKICAgICAgICB6ID0gdG9yY2guemVyb3MoMCwgZHR5cGU9c2lnbWFfZC5kdHlwZSkKICAgICAgICByZXR1cm4geiwgdG9yY2guemVyb3MoZCwgMCwgZHR5cGU9c2lnbWFfZC5kdHlwZSksIHRyYWNlX2QsIDAuMAogICAgZWxzZToKICAgICAgICAjIEJMQVMtMyB3b3JrIGdvZXMgdG8gdGhlIEdQVSBpbiBmbG9hdDMyOyB0aGUgY292YXJpYW5jZSB3YXMgYWNjdW11bGF0ZWQKICAgICAgICAjIGluIGZsb2F0MzIgYW55d2F5LCBzbyB0aGlzIGNvc3RzIG5vIHJlYWwgcHJlY2lzaW9uLgogICAgICAgIGlmIGRldmljZSBpcyBub3QgTm9uZSBhbmQgZGV2aWNlLnR5cGUgPT0gImN1ZGEiOgogICAgICAgICAgICBzZCA9IHNpZ21hX2QudG8oZGV2aWNlPWRldmljZSwgZHR5cGU9dG9yY2guZmxvYXQzMikKICAgICAgICAgICAgdiA9IHZfY29tcC50byhkZXZpY2U9ZGV2aWNlLCBkdHlwZT10b3JjaC5mbG9hdDMyKQogICAgICAgICAgICBtID0gKHYuVCBAIChzZCBAIHYpKS5kb3VibGUoKS5jcHUoKQogICAgICAgICAgICBkZWwgc2QsIHYKICAgICAgICAgICAgdG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgbSA9IHZfY29tcC5UIEAgKHNpZ21hX2QgQCB2X2NvbXApCiAgICAgICAgYmFjayA9IHZfY29tcAoKICAgIG0gPSAwLjUgKiAobSArIG0uVCkKICAgIGV2YWxzLCBldmVjcyA9IHRvcmNoLmxpbmFsZy5laWdoKG0pCiAgICBldmFscyA9IHRvcmNoLmZsaXAoZXZhbHMsIFswXSkuY2xhbXBfbWluKDApCiAgICBldmVjcyA9IHRvcmNoLmZsaXAoZXZlY3MsIFsxXSkKICAgIHIgPSBtaW4ocl9rZWVwLCBldmVjcy5zaGFwZVsxXSkKICAgIHRvcCA9IGV2ZWNzWzosIDpyXQogICAgaWYgYmFjayBpcyBub3QgTm9uZToKICAgICAgICB0b3AgPSBiYWNrLnRvKHRvcC5kdHlwZSkgQCB0b3AgICAgICAgIyBtYXAgYmFjayB0byB0aGUgb3JpZ2luYWwgc3BhY2UKICAgIHJldHVybiBldmFscywgdG9wLmNvbnRpZ3VvdXMoKSwgdHJhY2VfZCwgZmxvYXQoZXZhbHMuc3VtKCkpCgoKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQojIGJ1ZGdldGVkIHJhbmsgYWxsb2NhdGlvbgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCmRlZiBhbGxvY2F0ZV9yYW5rcyhzcGVjdHJhLCBjb3N0cywgYnVkZ2V0X3BhcmFtcywgcl9taW49MCwgcl9tYXg9NjQsCiAgICAgICAgICAgICAgICAgICBzY29yZV9tb2RlPSJyZWxhdGl2ZSIsIG5vcm1zPU5vbmUsIHNlbnNpdGl2aXR5PU5vbmUpOgogICAgIiIiR3JlZWR5IG1hcmdpbmFsLWdhaW4gYWxsb2NhdGlvbiBvZiBhIGdsb2JhbCBwYXJhbWV0ZXIgYnVkZ2V0LgoKICAgIHNwZWN0cmEgICA6IHtuYW1lOiAxLUQgZGVzY2VuZGluZyBhcnJheSBvZiBkcmlmdCBlaWdlbnZhbHVlc30KICAgIGNvc3RzICAgICA6IHtuYW1lOiBwYXJhbWV0ZXJzIGNvbnN1bWVkIHBlciB1bml0IHJhbmt9CiAgICBub3JtcyAgICAgOiB7bmFtZTogdHJhY2UoU2lnbWFfRCl9IHVzZWQgd2hlbiBzY29yZV9tb2RlID09ICJyZWxhdGl2ZSIKICAgIGJ1ZGdldCAgICA6IHRvdGFsIGFkYXB0ZXIgcGFyYW1ldGVycyBhdmFpbGFibGUKCiAgICBPYmplY3RpdmU6ICBtYXggIHN1bV9tIHdfbSAqIHN1bV97aTw9cl9tfSBsYW1iZGFfaGF0X3ttLGl9CiAgICAgICAgICAgICAgICBzLnQuIHN1bV9tIHJfbSAqIGNfbSA8PSBCLgoKICAgIEVhY2ggcGVyLW1vZHVsZSB2YWx1ZSBmdW5jdGlvbiBpcyBjb25jYXZlIGluIHJfbSBiZWNhdXNlIHRoZSBlaWdlbnZhbHVlcyBhcmUKICAgIHNvcnRlZCBkZXNjZW5kaW5nLCBzbyB0aGlzIHNlcGFyYWJsZSBjb25jYXZlIGtuYXBzYWNrIGlzIHNvbHZlZCBleGFjdGx5IGJ5CiAgICBncmVlZHkgbWFyZ2luYWwtdmFsdWUtcGVyLXBhcmFtZXRlciBzZWxlY3Rpb24gLS0gbm8gc2VhcmNoLCBubyB0cmFpbmluZy4KICAgICIiIgogICAgdmFscyA9IHt9CiAgICBmb3IgbmFtZSwgZXYgaW4gc3BlY3RyYS5pdGVtcygpOgogICAgICAgIGV2ID0gbnAuYXNhcnJheShldiwgZHR5cGU9bnAuZmxvYXQ2NCkKICAgICAgICBpZiBzY29yZV9tb2RlID09ICJyZWxhdGl2ZSI6CiAgICAgICAgICAgIGRlbm9tID0gZmxvYXQobm9ybXNbbmFtZV0pIGlmIG5vcm1zIGFuZCBub3Jtcy5nZXQobmFtZSwgMCkgPiAwIGVsc2UgbWF4KGV2LnN1bSgpLCAxZS0xMikKICAgICAgICAgICAgdiA9IGV2IC8gZGVub20KICAgICAgICBlbHNlOgogICAgICAgICAgICB2ID0gZXYKICAgICAgICBpZiBzZW5zaXRpdml0eSBpcyBub3QgTm9uZToKICAgICAgICAgICAgdiA9IHYgKiBmbG9hdChzZW5zaXRpdml0eS5nZXQobmFtZSwgMS4wKSkKICAgICAgICB2YWxzW25hbWVdID0gdgoKICAgIHJhbmtzID0ge246IDAgZm9yIG4gaW4gc3BlY3RyYX0KICAgIHNwZW50ID0gMAogICAgaWYgcl9taW4gPiAwOgogICAgICAgIGZvciBuIGluIHNwZWN0cmE6CiAgICAgICAgICAgIGsgPSBtaW4ocl9taW4sIGxlbih2YWxzW25dKSwgcl9tYXgpCiAgICAgICAgICAgIHJhbmtzW25dID0gawogICAgICAgICAgICBzcGVudCArPSBrICogY29zdHNbbl0KCiAgICBoZWFwID0gW10KICAgIGZvciBuIGluIHNwZWN0cmE6CiAgICAgICAgciA9IHJhbmtzW25dCiAgICAgICAgaWYgciA8IG1pbihyX21heCwgbGVuKHZhbHNbbl0pKToKICAgICAgICAgICAgaGVhcHEuaGVhcHB1c2goaGVhcCwgKC12YWxzW25dW3JdIC8gY29zdHNbbl0sIG4sIHIpKQogICAgd2hpbGUgaGVhcCBhbmQgc3BlbnQgPCBidWRnZXRfcGFyYW1zOgogICAgICAgIF8sIG4sIHIgPSBoZWFwcS5oZWFwcG9wKGhlYXApCiAgICAgICAgaWYgcmFua3Nbbl0gIT0gcjoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBpZiBzcGVudCArIGNvc3RzW25dID4gYnVkZ2V0X3BhcmFtczoKICAgICAgICAgICAgYnJlYWsKICAgICAgICByYW5rc1tuXSA9IHIgKyAxCiAgICAgICAgc3BlbnQgKz0gY29zdHNbbl0KICAgICAgICBpZiByICsgMSA8IG1pbihyX21heCwgbGVuKHZhbHNbbl0pKToKICAgICAgICAgICAgaGVhcHEuaGVhcHB1c2goaGVhcCwgKC12YWxzW25dW3IgKyAxXSAvIGNvc3RzW25dLCBuLCByICsgMSkpCiAgICByZXR1cm4gcmFua3MsIHNwZW50CgoKZGVmIHVuaWZvcm1fcmFua3MobW9kdWxlcywgYnVkZ2V0X3BhcmFtcywgcl9jYXA9Tm9uZSk6CiAgICAiIiJMYXJnZXN0IHVuaWZvcm0gcmFuayBmaXR0aW5nIHRoZSBidWRnZXQgKHRoZSBtYXRjaGVkLWJ1ZGdldCBMb1JBIGJhc2VsaW5lKS4iIiIKICAgIHRvdGFsX2Nvc3QgPSBzdW0obW9kdWxlX2Nvc3QobSkgZm9yIG0gaW4gbW9kdWxlcy52YWx1ZXMoKSkKICAgIHIgPSBpbnQoYnVkZ2V0X3BhcmFtcyAvLyB0b3RhbF9jb3N0KQogICAgaWYgcl9jYXAgaXMgbm90IE5vbmU6CiAgICAgICAgciA9IG1pbihyLCByX2NhcCkKICAgIHIgPSBtYXgociwgMSkKICAgIHJldHVybiB7bjogciBmb3IgbiBpbiBtb2R1bGVzfSwgciAqIHRvdGFsX2Nvc3QK", "peft_methods.py": "IiIiVW5pZmllZCBpbXBsZW1lbnRhdGlvbnMgb2YgdGhlIFBFRlQgbWV0aG9kcyBjb21wYXJlZCBpbiB0aGUgcGFwZXIuCgpFdmVyeXRoaW5nIGlzIGltcGxlbWVudGVkIGluc2lkZSBvbmUgZnJhbWV3b3JrIHNvIHRoYXQgdHJhaW5hYmxlLXBhcmFtZXRlciBidWRnZXRzLApvcHRpbWlzZXIgc2V0dGluZ3MgYW5kIHRyYWluaW5nIGNvZGUgYXJlICppZGVudGljYWwqIGFjcm9zcyBtZXRob2RzOyBvbmx5IHRoZQphZGFwdGVyIHBhcmFtZXRlcmlzYXRpb24gYW5kIGl0cyByYW5rIGFsbG9jYXRpb24gLyBpbml0aWFsaXNhdGlvbiBkaWZmZXIuCgpNZXRob2RzCi0tLS0tLS0KZnVsbCAgICAgOiBmdWxsIGZpbmUtdHVuaW5nIChyZWZlcmVuY2UgdXBwZXIgYm91bmQgb24gdHJhaW5hYmxlIHBhcmFtZXRlcnMpCmxpbmVhciAgIDogbGluZWFyIHByb2JlIC0tIGNsYXNzaWZpY2F0aW9uIGhlYWQgb25seQpiaXRmaXQgICA6IGFsbCBiaWFzIHRlcm1zICsgaGVhZCAgICAgICAgICAgICAgICAgICAgICAoQmVuIFpha2VuIGV0IGFsLiwgMjAyMikKbG9yYSAgICAgOiB1bmlmb3JtIHJhbmsgYWNyb3NzIG1vZHVsZXMgICAgICAgICAgICAgICAgKEh1IGV0IGFsLiwgMjAyMikKZG9yYSAgICAgOiB3ZWlnaHQtZGVjb21wb3NlZCBMb1JBICAgICAgICAgICAgICAgICAgICAgKExpdSBldCBhbC4sIDIwMjQpCnBpc3NhICAgIDogTG9SQSBpbml0aWFsaXNlZCBmcm9tIHRoZSB0b3AtciBTVkQgb2YgVyAgIChNZW5nIGV0IGFsLiwgMjAyNCkKYWRhbG9yYSAgOiBTVkQgcGFyYW1ldGVyaXNhdGlvbiArIHRyYWluaW5nLXRpbWUgaW1wb3J0YW5jZSBwcnVuaW5nIChaaGFuZyBldCBhbC4sIDIwMjMpCmV2YSAgICAgIDogaW4tZG9tYWluIGFjdGl2YXRpb24gUENBIGluaXQgKyBleHBsYWluZWQtdmFyaWFuY2UgcmFuayByZWRpc3RyaWJ1dGlvbgogICAgICAgICAgIChQYWlzY2hlciBldCBhbC4sIDIwMjUpOyBleGFjdGx5IHRoZSB0YXUgPSAwIGNhc2Ugb2YgZHJpZnQKZHJpZnQgICAgOiBvdXJzIC0tIHJlZmVyZW5jZS1jb250cmFzdGl2ZSBkcmlmdCBzdWJzcGFjZSAoc2VlIGRyaWZ0LnB5KQoiIiIKaW1wb3J0IG1hdGgKaW1wb3J0IHJlCmltcG9ydCB0b3JjaAppbXBvcnQgdG9yY2gubm4gYXMgbm4KaW1wb3J0IHRvcmNoLm5uLmZ1bmN0aW9uYWwgYXMgRgoKCiMgPT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT0KIyBhZGFwdGVyIGxheWVycwojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CmNsYXNzIExvUkFMaW5lYXIobm4uTW9kdWxlKToKICAgICIiIkZyb3plbiBiYXNlIGxpbmVhciArIHRyYWluYWJsZSBsb3ctcmFuayB1cGRhdGUgKG9wdGlvbmFsbHkgRG9SQS1zdHlsZSkuIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGJhc2U6IG5uLkxpbmVhciwgcjogaW50LCBhbHBoYTogZmxvYXQsIGRyb3BvdXQ6IGZsb2F0ID0gMC4wLAogICAgICAgICAgICAgICAgIHVzZV9kb3JhOiBib29sID0gRmFsc2UsIHNjYWxpbmdfbW9kZTogc3RyID0gImFscGhhX292ZXJfciIpOgogICAgICAgIHN1cGVyKCkuX19pbml0X18oKQogICAgICAgIHNlbGYuYmFzZSA9IGJhc2UKICAgICAgICBzZWxmLmJhc2Uud2VpZ2h0LnJlcXVpcmVzX2dyYWRfKEZhbHNlKQogICAgICAgIGlmIHNlbGYuYmFzZS5iaWFzIGlzIG5vdCBOb25lOgogICAgICAgICAgICBzZWxmLmJhc2UuYmlhcy5yZXF1aXJlc19ncmFkXyhGYWxzZSkKICAgICAgICBzZWxmLnIgPSBpbnQocikKICAgICAgICBzZWxmLnVzZV9kb3JhID0gdXNlX2RvcmEKICAgICAgICBzZWxmLmRyb3AgPSBubi5Ecm9wb3V0KGRyb3BvdXQpIGlmIGRyb3BvdXQgPiAwIGVsc2Ugbm4uSWRlbnRpdHkoKQogICAgICAgIGlmIHNlbGYuciA+IDA6CiAgICAgICAgICAgIHNlbGYubG9yYV9BID0gbm4uUGFyYW1ldGVyKHRvcmNoLnplcm9zKHNlbGYuciwgYmFzZS5pbl9mZWF0dXJlcykpCiAgICAgICAgICAgIHNlbGYubG9yYV9CID0gbm4uUGFyYW1ldGVyKHRvcmNoLnplcm9zKGJhc2Uub3V0X2ZlYXR1cmVzLCBzZWxmLnIpKQogICAgICAgICAgICBubi5pbml0LmthaW1pbmdfdW5pZm9ybV8oc2VsZi5sb3JhX0EsIGE9bWF0aC5zcXJ0KDUpKQogICAgICAgICAgICBpZiBzY2FsaW5nX21vZGUgPT0gIm9uZSI6CiAgICAgICAgICAgICAgICBzZWxmLnNjYWxpbmcgPSAxLjAKICAgICAgICAgICAgZWxpZiBzY2FsaW5nX21vZGUgPT0gInJzbG9yYSI6CiAgICAgICAgICAgICAgICBzZWxmLnNjYWxpbmcgPSBhbHBoYSAvIG1hdGguc3FydChzZWxmLnIpCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBzZWxmLnNjYWxpbmcgPSBhbHBoYSAvIHNlbGYucgogICAgICAgICAgICBpZiB1c2VfZG9yYToKICAgICAgICAgICAgICAgIHdpdGggdG9yY2gubm9fZ3JhZCgpOgogICAgICAgICAgICAgICAgICAgIG0gPSB0b3JjaC5saW5hbGcubm9ybShiYXNlLndlaWdodCwgZGltPTEpCiAgICAgICAgICAgICAgICBzZWxmLmRvcmFfbSA9IG5uLlBhcmFtZXRlcihtKQogICAgICAgIGVsc2U6CiAgICAgICAgICAgIHNlbGYuc2NhbGluZyA9IDAuMAoKICAgIGRlZiBkZWx0YV93KHNlbGYpOgogICAgICAgIHJldHVybiAoc2VsZi5sb3JhX0IgQCBzZWxmLmxvcmFfQSkgKiBzZWxmLnNjYWxpbmcKCiAgICBkZWYgZm9yd2FyZChzZWxmLCB4KToKICAgICAgICBpZiBzZWxmLnIgPT0gMDoKICAgICAgICAgICAgcmV0dXJuIHNlbGYuYmFzZSh4KQogICAgICAgIGlmIG5vdCBzZWxmLnVzZV9kb3JhOgogICAgICAgICAgICBvdXQgPSBzZWxmLmJhc2UoeCkKICAgICAgICAgICAgaCA9IHNlbGYuZHJvcCh4KSBAIHNlbGYubG9yYV9BLlQKICAgICAgICAgICAgcmV0dXJuIG91dCArIChoIEAgc2VsZi5sb3JhX0IuVCkgKiBzZWxmLnNjYWxpbmcKICAgICAgICB3ID0gc2VsZi5iYXNlLndlaWdodCArIHNlbGYuZGVsdGFfdygpCiAgICAgICAgbm9ybSA9IHRvcmNoLmxpbmFsZy5ub3JtKHcsIGRpbT0xKS5jbGFtcF9taW4oMWUtOCkuZGV0YWNoKCkKICAgICAgICB3ID0gdyAqIChzZWxmLmRvcmFfbSAvIG5vcm0pLnVuc3F1ZWV6ZSgxKQogICAgICAgIHJldHVybiBGLmxpbmVhcihzZWxmLmRyb3AoeCksIHcsIHNlbGYuYmFzZS5iaWFzKQoKCmNsYXNzIEFkYUxvUkFMaW5lYXIobm4uTW9kdWxlKToKICAgICIiIlNWRC1zdHlsZSBwYXJhbWV0ZXJpc2F0aW9uIGRXID0gUCBkaWFnKEUpIFEgdXNlZCBieSBBZGFMb1JBLgoKICAgIFRyaXBsZXRzIGFyZSBtYXNrZWQgKEVfaSA8LSAwKSBieSB0aGUgZ2xvYmFsIGJ1ZGdldCBjb250cm9sbGVyOyBtYXNrZWQgdHJpcGxldHMKICAgIHN0b3AgY29udHJpYnV0aW5nIGJ1dCBzdGF5IGFsbG9jYXRlZCB1bnRpbCB0aGUgc2NoZWR1bGUgZW5kcywgZXhhY3RseSBhcyBpbiB0aGUKICAgIG9yaWdpbmFsIGZvcm11bGF0aW9uLgogICAgIiIiCgogICAgZGVmIF9faW5pdF9fKHNlbGYsIGJhc2U6IG5uLkxpbmVhciwgcjogaW50LCBhbHBoYTogZmxvYXQsIGRyb3BvdXQ6IGZsb2F0ID0gMC4wKToKICAgICAgICBzdXBlcigpLl9faW5pdF9fKCkKICAgICAgICBzZWxmLmJhc2UgPSBiYXNlCiAgICAgICAgc2VsZi5iYXNlLndlaWdodC5yZXF1aXJlc19ncmFkXyhGYWxzZSkKICAgICAgICBpZiBzZWxmLmJhc2UuYmlhcyBpcyBub3QgTm9uZToKICAgICAgICAgICAgc2VsZi5iYXNlLmJpYXMucmVxdWlyZXNfZ3JhZF8oRmFsc2UpCiAgICAgICAgc2VsZi5yID0gaW50KHIpCiAgICAgICAgc2VsZi5kcm9wID0gbm4uRHJvcG91dChkcm9wb3V0KSBpZiBkcm9wb3V0ID4gMCBlbHNlIG5uLklkZW50aXR5KCkKICAgICAgICBzZWxmLmxvcmFfUCA9IG5uLlBhcmFtZXRlcih0b3JjaC56ZXJvcyhiYXNlLm91dF9mZWF0dXJlcywgc2VsZi5yKSkKICAgICAgICBzZWxmLmxvcmFfRSA9IG5uLlBhcmFtZXRlcih0b3JjaC56ZXJvcyhzZWxmLnIpKQogICAgICAgIHNlbGYubG9yYV9RID0gbm4uUGFyYW1ldGVyKHRvcmNoLnplcm9zKHNlbGYuciwgYmFzZS5pbl9mZWF0dXJlcykpCiAgICAgICAgbm4uaW5pdC5ub3JtYWxfKHNlbGYubG9yYV9QLCBzdGQ9MC4wMikKICAgICAgICBubi5pbml0Lm5vcm1hbF8oc2VsZi5sb3JhX1EsIHN0ZD0wLjAyKQogICAgICAgIG5uLmluaXQuemVyb3NfKHNlbGYubG9yYV9FKQogICAgICAgIHNlbGYuc2NhbGluZyA9IGFscGhhIC8gc2VsZi5yCiAgICAgICAgc2VsZi5yZWdpc3Rlcl9idWZmZXIoIm1hc2siLCB0b3JjaC5vbmVzKHNlbGYucikpCgogICAgZGVmIGZvcndhcmQoc2VsZiwgeCk6CiAgICAgICAgb3V0ID0gc2VsZi5iYXNlKHgpCiAgICAgICAgZSA9IHNlbGYubG9yYV9FICogc2VsZi5tYXNrCiAgICAgICAgaCA9IHNlbGYuZHJvcCh4KSBAIHNlbGYubG9yYV9RLlQKICAgICAgICBoID0gaCAqIGUKICAgICAgICByZXR1cm4gb3V0ICsgKGggQCBzZWxmLmxvcmFfUC5UKSAqIHNlbGYuc2NhbGluZwoKICAgIGRlZiBvcnRob19wZW5hbHR5KHNlbGYpOgogICAgICAgIHAsIHEgPSBzZWxmLmxvcmFfUCwgc2VsZi5sb3JhX1EKICAgICAgICBpcCA9IHAuVCBAIHAKICAgICAgICBpcSA9IHEgQCBxLlQKICAgICAgICBleWUgPSB0b3JjaC5leWUoc2VsZi5yLCBkZXZpY2U9cC5kZXZpY2UsIGR0eXBlPXAuZHR5cGUpCiAgICAgICAgcmV0dXJuICgoaXAgLSBleWUpICoqIDIpLnN1bSgpICsgKChpcSAtIGV5ZSkgKiogMikuc3VtKCkKCgojID09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09CiMgaW5qZWN0aW9uIGhlbHBlcnMKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQpkZWYgX2dldF9wYXJlbnQobW9kZWwsIG5hbWUpOgogICAgcGFydHMgPSBuYW1lLnNwbGl0KCIuIikKICAgIHBhcmVudCA9IG1vZGVsCiAgICBmb3IgcCBpbiBwYXJ0c1s6LTFdOgogICAgICAgIHBhcmVudCA9IGdldGF0dHIocGFyZW50LCBwKQogICAgcmV0dXJuIHBhcmVudCwgcGFydHNbLTFdCgoKZGVmIGluamVjdF9hZGFwdGVycyhtb2RlbCwgcmFua3MsIGFscGhhPTE2LjAsIGRyb3BvdXQ9MC4wLCBraW5kPSJsb3JhIiwKICAgICAgICAgICAgICAgICAgICBzY2FsaW5nX21vZGU9ImFscGhhX292ZXJfciIpOgogICAgIiIiUmVwbGFjZSB0aGUgbmFtZWQgbm4uTGluZWFyIG1vZHVsZXMgd2l0aCBhZGFwdGVyLXdyYXBwZWQgdmVyc2lvbnMuIiIiCiAgICBpbmplY3RlZCA9IHt9CiAgICBmb3IgbmFtZSwgciBpbiByYW5rcy5pdGVtcygpOgogICAgICAgIGlmIHIgPD0gMDoKICAgICAgICAgICAgY29udGludWUKICAgICAgICBwYXJlbnQsIGF0dHIgPSBfZ2V0X3BhcmVudChtb2RlbCwgbmFtZSkKICAgICAgICBiYXNlID0gZ2V0YXR0cihwYXJlbnQsIGF0dHIpCiAgICAgICAgaWYga2luZCA9PSAiYWRhbG9yYSI6CiAgICAgICAgICAgIG5ldyA9IEFkYUxvUkFMaW5lYXIoYmFzZSwgciwgYWxwaGEsIGRyb3BvdXQpCiAgICAgICAgZWxzZToKICAgICAgICAgICAgbmV3ID0gTG9SQUxpbmVhcihiYXNlLCByLCBhbHBoYSwgZHJvcG91dCwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICB1c2VfZG9yYT0oa2luZCA9PSAiZG9yYSIpLCBzY2FsaW5nX21vZGU9c2NhbGluZ19tb2RlKQogICAgICAgIHNldGF0dHIocGFyZW50LCBhdHRyLCBuZXcpCiAgICAgICAgaW5qZWN0ZWRbbmFtZV0gPSBuZXcKICAgIHJldHVybiBpbmplY3RlZAoKCmRlZiBmcmVlemVfYmFja2JvbmUobW9kZWwsIGhlYWRfcHJlZml4ZXM9KCJjbGFzc2lmaWVyIiwgInNjb3JlIiwgImhlYWQiKSk6CiAgICBmb3IgbmFtZSwgcCBpbiBtb2RlbC5uYW1lZF9wYXJhbWV0ZXJzKCk6CiAgICAgICAgcC5yZXF1aXJlc19ncmFkXyhhbnkobmFtZS5zdGFydHN3aXRoKGgpIG9yICgiLiIgKyBoICsgIi4iKSBpbiBuYW1lCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9yIGggaW4gaGVhZF9wcmVmaXhlcykpCgoKZGVmIHVuZnJlZXplX2hlYWQobW9kZWwsIGhlYWRfcHJlZml4ZXM9KCJjbGFzc2lmaWVyIiwgInNjb3JlIiwgImhlYWQiKSk6CiAgICBmb3IgbmFtZSwgcCBpbiBtb2RlbC5uYW1lZF9wYXJhbWV0ZXJzKCk6CiAgICAgICAgaWYgYW55KGggaW4gbmFtZS5zcGxpdCgiLiIpIGZvciBoIGluIGhlYWRfcHJlZml4ZXMpOgogICAgICAgICAgICBwLnJlcXVpcmVzX2dyYWRfKFRydWUpCgoKZGVmIHNldF90cmFpbmFibGVfYWRhcHRlcnMobW9kZWwpOgogICAgZm9yIG5hbWUsIHAgaW4gbW9kZWwubmFtZWRfcGFyYW1ldGVycygpOgogICAgICAgIGlmIGFueShrIGluIG5hbWUgZm9yIGsgaW4gKCJsb3JhX0EiLCAibG9yYV9CIiwgImxvcmFfUCIsICJsb3JhX0UiLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICJsb3JhX1EiLCAiZG9yYV9tIikpOgogICAgICAgICAgICBwLnJlcXVpcmVzX2dyYWRfKFRydWUpCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojIGluaXRpYWxpc2F0aW9uIHNjaGVtZXMKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQpAdG9yY2gubm9fZ3JhZCgpCmRlZiBpbml0X3Bpc3NhKGxheWVyOiBMb1JBTGluZWFyKToKICAgICIiIkEgPSBzcXJ0KFNfcikgVl9yXlQsIEIgPSBVX3Igc3FydChTX3IpOyByZXNpZHVhbCB3ZWlnaHQgVyAtIEJBIHN0YXlzIGZyb3plbi4iIiIKICAgIHcgPSBsYXllci5iYXNlLndlaWdodC5kYXRhLmRvdWJsZSgpCiAgICB1LCBzLCB2aCA9IHRvcmNoLmxpbmFsZy5zdmQodywgZnVsbF9tYXRyaWNlcz1GYWxzZSkKICAgIHIgPSBsYXllci5yCiAgICB1ciwgc3IsIHZyID0gdVs6LCA6cl0sIHNbOnJdLCB2aFs6ciwgOl0KICAgIHNxID0gdG9yY2guc3FydChzcikKICAgIGEgPSAodG9yY2guZGlhZyhzcSkgQCB2cikKICAgIGIgPSAodXIgQCB0b3JjaC5kaWFnKHNxKSkKICAgIGxheWVyLmxvcmFfQS5kYXRhLmNvcHlfKGEudG8obGF5ZXIubG9yYV9BLmR0eXBlKSkKICAgIGxheWVyLmxvcmFfQi5kYXRhLmNvcHlfKGIudG8obGF5ZXIubG9yYV9CLmR0eXBlKSkKICAgIGxheWVyLnNjYWxpbmcgPSAxLjAKICAgIGxheWVyLmJhc2Uud2VpZ2h0LmRhdGEuY29weV8oKHcgLSBiIEAgYSkudG8obGF5ZXIuYmFzZS53ZWlnaHQuZHR5cGUpKQoKCkB0b3JjaC5ub19ncmFkKCkKZGVmIGluaXRfc3Vic3BhY2UobGF5ZXI6IExvUkFMaW5lYXIsIGJhc2lzOiB0b3JjaC5UZW5zb3IpOgogICAgIiIiQSA8LSBsZWFkaW5nIHN1YnNwYWNlIGRpcmVjdGlvbnMgKHJvd3MpLCBCIDwtIDAgc28gZFcgPSAwIGF0IGluaXRpYWxpc2F0aW9uLgoKICAgIGJhc2lzOiAoZF9pbiwgaykgY29sdW1uLW9ydGhvbm9ybWFsLCBrID49IGxheWVyLnIgaW4gbm9ybWFsIG9wZXJhdGlvbi4KCiAgICBJZiB0aGUgY2FjaGVkIGJhc2lzIGhhcyBmZXdlciBjb2x1bW5zIHRoYW4gdGhlIGFsbG9jYXRlZCByYW5rLCB0aGUgc3VycGx1cwogICAgcm93cyBhcmUgZmlsbGVkIHdpdGggcmFuZG9tIGRpcmVjdGlvbnMgb3J0aG9nb25hbGlzZWQgYWdhaW5zdCB0aGUgYmFzaXMgLS0KICAgIG5ldmVyIGxlZnQgYXMgemVyb3MsIHdoaWNoIHdvdWxkIG1ha2UgdGhvc2UgcmFua3MgcGVybWFuZW50bHkgZGVhZCAoYSB6ZXJvCiAgICByb3cgb2YgQSBnaXZlcyBhIHplcm8gZ3JhZGllbnQgdG8gdGhlIGNvcnJlc3BvbmRpbmcgY29sdW1uIG9mIEIpLgogICAgIiIiCiAgICByID0gbWluKGxheWVyLnIsIGJhc2lzLnNoYXBlWzFdKQogICAgbGF5ZXIubG9yYV9BLmRhdGEuemVyb18oKQogICAgbGF5ZXIubG9yYV9BLmRhdGFbOnJdLmNvcHlfKGJhc2lzWzosIDpyXS5ULnRvKGxheWVyLmxvcmFfQS5kdHlwZSkpCiAgICBpZiBsYXllci5yID4gcjoKICAgICAgICBleHRyYSA9IHRvcmNoLnJhbmRuKGxheWVyLmJhc2UuaW5fZmVhdHVyZXMsIGxheWVyLnIgLSByLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgZHR5cGU9YmFzaXMuZHR5cGUpCiAgICAgICAgZXh0cmEgLT0gYmFzaXNbOiwgOnJdIEAgKGJhc2lzWzosIDpyXS5UIEAgZXh0cmEpCiAgICAgICAgcSwgXyA9IHRvcmNoLmxpbmFsZy5xcihleHRyYSkKICAgICAgICBsYXllci5sb3JhX0EuZGF0YVtyOl0uY29weV8ocS5ULnRvKGxheWVyLmxvcmFfQS5kdHlwZSkpCiAgICBsYXllci5sb3JhX0IuZGF0YS56ZXJvXygpCgoKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQojIEFkYUxvUkEgYnVkZ2V0IGNvbnRyb2xsZXIKIyA9PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PT09PQpjbGFzcyBBZGFMb1JBQ29udHJvbGxlcjoKICAgICIiIkdsb2JhbCBpbXBvcnRhbmNlLWJhc2VkIGJ1ZGdldCBzY2hlZHVsZXIgKFpoYW5nIGV0IGFsLiwgSUNMUiAyMDIzKS4KCiAgICBJbXBvcnRhbmNlIG9mIHRyaXBsZXQgaSBjb21iaW5lcyB0aGUgc2Vuc2l0aXZpdHkgb2YgRV9pIGFuZCBvZiB0aGUgY29ycmVzcG9uZGluZwogICAgcm93L2NvbHVtbiBvZiBQIGFuZCBRLCBlYWNoIHNtb290aGVkIGJ5IGFuIGV4cG9uZW50aWFsIG1vdmluZyBhdmVyYWdlLCBwbHVzIGFuCiAgICB1bmNlcnRhaW50eSB0ZXJtLiAgVGhlIHRvdGFsIGJ1ZGdldCBmb2xsb3dzIGEgY3ViaWMgc2NoZWR1bGUgZnJvbSBiX2luaXQgdG8KICAgIGJfdGFyZ2V0OyB0aGUgbG93ZXN0LWltcG9ydGFuY2UgdHJpcGxldHMgYXJlIG1hc2tlZC4KICAgICIiIgoKICAgIGRlZiBfX2luaXRfXyhzZWxmLCBsYXllcnMsIHRhcmdldF9yYW5rX3RvdGFsLCBpbml0X3JhbmtfdG90YWwsCiAgICAgICAgICAgICAgICAgdG90YWxfc3RlcHMsIHdhcm11cF9mcmFjPTAuMSwgZmluYWxfZnJhYz0wLjc1LAogICAgICAgICAgICAgICAgIGJldGExPTAuODUsIGJldGEyPTAuODUpOgogICAgICAgIHNlbGYubGF5ZXJzID0gbGF5ZXJzCiAgICAgICAgc2VsZi5iX3RhcmdldCA9IHRhcmdldF9yYW5rX3RvdGFsCiAgICAgICAgc2VsZi5iX2luaXQgPSBpbml0X3JhbmtfdG90YWwKICAgICAgICBzZWxmLnRpID0gaW50KHdhcm11cF9mcmFjICogdG90YWxfc3RlcHMpCiAgICAgICAgc2VsZi50ZiA9IGludChmaW5hbF9mcmFjICogdG90YWxfc3RlcHMpCiAgICAgICAgc2VsZi50b3RhbF9zdGVwcyA9IHRvdGFsX3N0ZXBzCiAgICAgICAgc2VsZi5iZXRhMSwgc2VsZi5iZXRhMiA9IGJldGExLCBiZXRhMgogICAgICAgIHNlbGYuaXB0LCBzZWxmLmV4cF9pcHQsIHNlbGYuZXhwX3VuYyA9IHt9LCB7fSwge30KCiAgICBkZWYgX3Njb3JlKHNlbGYsIGxheWVyLCBuYW1lKToKICAgICAgICB3aXRoIHRvcmNoLm5vX2dyYWQoKToKICAgICAgICAgICAgcGFydHMgPSBbXQogICAgICAgICAgICBmb3IgcCBpbiAobGF5ZXIubG9yYV9FLCBsYXllci5sb3JhX1AsIGxheWVyLmxvcmFfUSk6CiAgICAgICAgICAgICAgICBpZiBwLmdyYWQgaXMgTm9uZToKICAgICAgICAgICAgICAgICAgICByZXR1cm4gTm9uZQogICAgICAgICAgICAgICAgcyA9IChwICogcC5ncmFkKS5hYnMoKS5kZXRhY2goKQogICAgICAgICAgICAgICAgcGFydHMuYXBwZW5kKHMpCiAgICAgICAgICAgIGVfcyA9IHBhcnRzWzBdICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAjIChyLCkKICAgICAgICAgICAgcF9zID0gcGFydHNbMV0ubWVhbihkaW09MCkgICAgICAgICAgICAgICAgICAgICAgICMgKHIsKQogICAgICAgICAgICBxX3MgPSBwYXJ0c1syXS5tZWFuKGRpbT0xKSAgICAgICAgICAgICAgICAgICAgICAgIyAociwpCiAgICAgICAgICAgIHJhdyA9IGVfcyArIHBfcyArIHFfcwogICAgICAgICAgICBpZiBuYW1lIG5vdCBpbiBzZWxmLmV4cF9pcHQ6CiAgICAgICAgICAgICAgICBzZWxmLmV4cF9pcHRbbmFtZV0gPSB0b3JjaC56ZXJvc19saWtlKHJhdykKICAgICAgICAgICAgICAgIHNlbGYuZXhwX3VuY1tuYW1lXSA9IHRvcmNoLnplcm9zX2xpa2UocmF3KQogICAgICAgICAgICBzZWxmLmV4cF9pcHRbbmFtZV0gPSBzZWxmLmJldGExICogc2VsZi5leHBfaXB0W25hbWVdICsgKDEgLSBzZWxmLmJldGExKSAqIHJhdwogICAgICAgICAgICBzZWxmLmV4cF91bmNbbmFtZV0gPSAoc2VsZi5iZXRhMiAqIHNlbGYuZXhwX3VuY1tuYW1lXQogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgKyAoMSAtIHNlbGYuYmV0YTIpICogKHJhdyAtIHNlbGYuZXhwX2lwdFtuYW1lXSkuYWJzKCkpCiAgICAgICAgICAgIHJldHVybiBzZWxmLmV4cF9pcHRbbmFtZV0gKiBzZWxmLmV4cF91bmNbbmFtZV0KCiAgICBkZWYgYnVkZ2V0KHNlbGYsIHN0ZXApOgogICAgICAgIGlmIHN0ZXAgPD0gc2VsZi50aToKICAgICAgICAgICAgcmV0dXJuIHNlbGYuYl9pbml0CiAgICAgICAgaWYgc3RlcCA+PSBzZWxmLnRmOgogICAgICAgICAgICByZXR1cm4gc2VsZi5iX3RhcmdldAogICAgICAgIGZyYWMgPSAxLjAgLSAoc3RlcCAtIHNlbGYudGkpIC8gbWF4KHNlbGYudGYgLSBzZWxmLnRpLCAxKQogICAgICAgIHJldHVybiBpbnQoc2VsZi5iX3RhcmdldCArIChzZWxmLmJfaW5pdCAtIHNlbGYuYl90YXJnZXQpICogKGZyYWMgKiogMykpCgogICAgZGVmIHN0ZXAoc2VsZiwgZ2xvYmFsX3N0ZXApOgogICAgICAgIHNjb3JlcyA9IHt9CiAgICAgICAgZm9yIG5hbWUsIGxheWVyIGluIHNlbGYubGF5ZXJzLml0ZW1zKCk6CiAgICAgICAgICAgIHMgPSBzZWxmLl9zY29yZShsYXllciwgbmFtZSkKICAgICAgICAgICAgaWYgcyBpcyBOb25lOgogICAgICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgICAgIHNjb3Jlc1tuYW1lXSA9IHMKICAgICAgICBiID0gc2VsZi5idWRnZXQoZ2xvYmFsX3N0ZXApCiAgICAgICAgYWxsdiA9IHRvcmNoLmNhdChbdiBmb3IgdiBpbiBzY29yZXMudmFsdWVzKCldKQogICAgICAgIGsgPSBtYXgoaW50KGIpLCAxKQogICAgICAgIGlmIGsgPj0gYWxsdi5udW1lbCgpOgogICAgICAgICAgICBmb3IgbGF5ZXIgaW4gc2VsZi5sYXllcnMudmFsdWVzKCk6CiAgICAgICAgICAgICAgICBsYXllci5tYXNrLmZpbGxfKDEuMCkKICAgICAgICAgICAgcmV0dXJuCiAgICAgICAgdGhyZXNoID0gdG9yY2gudG9wayhhbGx2LCBrLCBsYXJnZXN0PVRydWUpLnZhbHVlcy5taW4oKQogICAgICAgIGZvciBuYW1lLCBsYXllciBpbiBzZWxmLmxheWVycy5pdGVtcygpOgogICAgICAgICAgICBsYXllci5tYXNrLmNvcHlfKChzY29yZXNbbmFtZV0gPj0gdGhyZXNoKS50byhsYXllci5tYXNrLmR0eXBlKSkKCiAgICBkZWYgYWN0aXZlX3JhbmtfdG90YWwoc2VsZik6CiAgICAgICAgcmV0dXJuIGludChzdW0obC5tYXNrLnN1bSgpLml0ZW0oKSBmb3IgbCBpbiBzZWxmLmxheWVycy52YWx1ZXMoKSkpCg==", "engine.py": "...", "profile_drift.py": "IiIiQ29tcHV0ZSBhbmQgY2FjaGUgRFJJRlQgcHJvZmlsZXMgZm9yIGEgKG1vZGVsLCB0YXNrKSBwYWlyLgoKT25lIGZvcndhcmQtb25seSBwYXNzIG92ZXIgYSBnZW5lcmFsLWRvbWFpbiByZWZlcmVuY2UgY29ycHVzIGFuZCBvbmUgb3ZlciB0aGUKdW5sYWJlbGxlZCB0YXNrIHRleHQgeWllbGRzLCBwZXIgYWRhcHRhYmxlIGxpbmVhciBtb2R1bGUsIHRoZSBzZWNvbmQtbW9tZW50Cm1hdHJpY2VzIFNpZ21hX0cgYW5kIFNpZ21hX0QuICBGb3IgZWFjaCByZXF1ZXN0ZWQgdGF1IHdlIHRoZW4gc3RvcmUKCiAgICAtIHRoZSBmdWxsIGRyaWZ0IHNwZWN0cnVtIChlaWdlbnZhbHVlcyBvZiBTaWdtYX4gaW4gZGVzY2VuZGluZyBvcmRlciksCiAgICAtIHRoZSBsZWFkaW5nIHJfbWF4IGRyaWZ0IGVpZ2VudmVjdG9ycyAodGhlIGFkYXB0ZXIgaW5pdGlhbGlzYXRpb24gYmFzaXMpLAogICAgLSB0cmFjZShTaWdtYV9EKSBhbmQgdGhlIHJlZmVyZW5jZSBzdWJzcGFjZSBkaW1lbnNpb24gay4KCnRhdSA9IDAgcmVkdWNlcyB0byBwbGFpbiBpbi1kb21haW4gYWN0aXZhdGlvbiBQQ0EsIGkuZS4gdGhlIEVWQSBiYXNlbGluZS4KClVzYWdlOgogICAgcHl0aG9uIHNyYy9wcm9maWxlX2RyaWZ0LnB5IC0tbW9kZWwgcm9iZXJ0YS1iYXNlIC0tdGFzayBjaGVtcHJvdAoiIiIKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgc3lzCmltcG9ydCB0aW1lCgppbXBvcnQgdG9yY2gKCnN5cy5wYXRoLmluc2VydCgwLCBvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSkpCmltcG9ydCBkYXRhIGFzIGRhdGFfbW9kICAgICAgICAgICMgbm9xYTogRTQwMgppbXBvcnQgZHJpZnQgYXMgZHJpZnRfbW9kICAgICAgICAjIG5vcWE6IEU0MDIKClJPT1QgPSBvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpKQpQUk9GSUxFX0RJUiA9IG9zLnBhdGguam9pbihST09ULCAicnVucyIsICJwcm9maWxlcyIpCgoKZGVmIHByb2ZpbGVfa2V5KG1vZGVsX25hbWUsIHRhc2ssIG5fcmVmLCBuX2RvbSwgY2VudGVyPVRydWUpOgogICAgc2FmZSA9IG1vZGVsX25hbWUucmVwbGFjZSgiLyIsICJfXyIpCiAgICAjIGNlbnRyZWQgKGNvdmFyaWFuY2UpIHByb2ZpbGVzIGFyZSB0aGUgZGVmYXVsdDsgdGhlIHN1ZmZpeCBrZWVwcyB0aGVtIGZyb20KICAgICMgZXZlciBiZWluZyBjb25mdXNlZCB3aXRoIHJhdyBzZWNvbmQtbW9tZW50IHByb2ZpbGVzIGNhY2hlZCBlYXJsaWVyCiAgICByZXR1cm4gZiJ7c2FmZX1fX3t0YXNrfV9fcmVme25fcmVmfV9fZG9te25fZG9tfSIgKyAoIl9fY2VuIiBpZiBjZW50ZXIgZWxzZSAiIikKCgpkZWYgbWFpbigpOgogICAgYXAgPSBhcmdwYXJzZS5Bcmd1bWVudFBhcnNlcigpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbW9kZWwiLCBkZWZhdWx0PSJyb2JlcnRhLWJhc2UiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXRhc2siLCBkZWZhdWx0PSJjaGVtcHJvdCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbl9yZWYiLCB0eXBlPWludCwgZGVmYXVsdD0xMDI0KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW5fZG9tIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MTAyNCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS10YXVzIiwgZGVmYXVsdD0iMC4wLDAuNSwwLjksMC45NSwwLjk5IikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1yX21heCIsIHR5cGU9aW50LCBkZWZhdWx0PTY0KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWJhdGNoX3NpemUiLCB0eXBlPWludCwgZGVmYXVsdD0xNikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1tYXhfbGVuIiwgdHlwZT1pbnQsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1mb3JjZSIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tb3V0X3N1ZmZpeCIsIGRlZmF1bHQ9IiIsCiAgICAgICAgICAgICAgICAgICAgaGVscD0id3JpdGUgdG8gYSBzZXBhcmF0ZWx5IG5hbWVkIHByb2ZpbGUsIGUuZy4gdG8gdGltZSBhICIKICAgICAgICAgICAgICAgICAgICAgICAgICJzaW5nbGUtdGF1IHJ1biB3aXRob3V0IHRvdWNoaW5nIHRoZSBjYWNoZWQgc3dlZXAiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXJhdyIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIsCiAgICAgICAgICAgICAgICAgICAgaGVscD0icmF3IHNlY29uZCBtb21lbnRzIGluc3RlYWQgb2YgY292YXJpYW5jZXMiKQogICAgYXJncyA9IGFwLnBhcnNlX2FyZ3MoKQoKICAgIGZyb20gdHJhbnNmb3JtZXJzIGltcG9ydCBBdXRvVG9rZW5pemVyLCBBdXRvTW9kZWxGb3JTZXF1ZW5jZUNsYXNzaWZpY2F0aW9uCgogICAgb3MubWFrZWRpcnMoUFJPRklMRV9ESVIsIGV4aXN0X29rPVRydWUpCiAgICBjZW50ZXIgPSBub3QgYXJncy5yYXcKICAgIGtleSA9IHByb2ZpbGVfa2V5KGFyZ3MubW9kZWwsIGFyZ3MudGFzaywgYXJncy5uX3JlZiwgYXJncy5uX2RvbSwKICAgICAgICAgICAgICAgICAgICAgIGNlbnRlcj1jZW50ZXIpICsgYXJncy5vdXRfc3VmZml4CiAgICBvdXRfcGF0aCA9IG9zLnBhdGguam9pbihQUk9GSUxFX0RJUiwga2V5ICsgIi5wdCIpCiAgICBpZiBvcy5wYXRoLmV4aXN0cyhvdXRfcGF0aCkgYW5kIG5vdCBhcmdzLmZvcmNlOgogICAgICAgIHByaW50KCJwcm9maWxlIGV4aXN0czoiLCBvdXRfcGF0aCkKICAgICAgICByZXR1cm4KCiAgICBkZXZpY2UgPSB0b3JjaC5kZXZpY2UoImN1ZGEiIGlmIHRvcmNoLmN1ZGEuaXNfYXZhaWxhYmxlKCkgZWxzZSAiY3B1IikKICAgIHRhc2sgPSBkYXRhX21vZC5sb2FkX3Rhc2soYXJncy50YXNrKQogICAgbWF4X2xlbiA9IGFyZ3MubWF4X2xlbiBvciBkYXRhX21vZC5UQVNLX01BWExFTi5nZXQoYXJncy50YXNrLCAxMjgpCgogICAgdG9rID0gQXV0b1Rva2VuaXplci5mcm9tX3ByZXRyYWluZWQoYXJncy5tb2RlbCkKICAgICMgcHJvZmlsZSBpbiBmcDMyIHdoYXRldmVyIHRoZSBjaGVja3BvaW50J3Mgc3RvcmVkIGR0eXBlIChiZjE2IGZvciBTbW9sTE0yKQogICAgbW9kZWwgPSBBdXRvTW9kZWxGb3JTZXF1ZW5jZUNsYXNzaWZpY2F0aW9uLmZyb21fcHJldHJhaW5lZCgKICAgICAgICBhcmdzLm1vZGVsLCBudW1fbGFiZWxzPXRhc2tbIm51bV9sYWJlbHMiXSkuZmxvYXQoKS50byhkZXZpY2UpCiAgICBkcmlmdF9tb2QuZW5zdXJlX3BhZGRpbmcodG9rLCBtb2RlbCkKICAgIG1vZGVsLmV2YWwoKQoKICAgIG1vZHVsZXMgPSBkcmlmdF9tb2QuZmluZF90YXJnZXRfbW9kdWxlcyhtb2RlbCkKICAgIHByaW50KGYie2xlbihtb2R1bGVzKX0gYWRhcHRhYmxlIG1vZHVsZXMiKQoKICAgIGRvbV90ZXh0cyA9IHRhc2tbInNwbGl0cyJdWyJ0cmFpbiJdWzBdWzphcmdzLm5fZG9tXQogICAgcmVmX3RleHRzID0gZGF0YV9tb2QubG9hZF9yZWZlcmVuY2VfY29ycHVzKG5fZG9jcz1hcmdzLm5fcmVmKQogICAgcHJpbnQoZiJyZWZlcmVuY2Uge2xlbihyZWZfdGV4dHMpfSBkb2NzIHwgZG9tYWluIHtsZW4oZG9tX3RleHRzKX0gZG9jcyB8IG1heF9sZW4ge21heF9sZW59IikKCiAgICB0MCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIGNvdl9nID0gZHJpZnRfbW9kLmNvbGxlY3RfY292YXJpYW5jZXMobW9kZWwsIHRvaywgcmVmX3RleHRzLCBtb2R1bGVzLCBkZXZpY2UsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIG1heF9sZW49bWF4X2xlbiwgYmF0Y2hfc2l6ZT1hcmdzLmJhdGNoX3NpemUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIGNlbnRlcj1jZW50ZXIpCiAgICB0X3JlZiA9IHRpbWUucGVyZl9jb3VudGVyKCkgLSB0MAogICAgcHJpbnQoZiJyZWZlcmVuY2UgcGFzcyB7dF9yZWY6LjFmfXMiKQoKICAgIHQxID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgY292X2QgPSBkcmlmdF9tb2QuY29sbGVjdF9jb3ZhcmlhbmNlcyhtb2RlbCwgdG9rLCBkb21fdGV4dHMsIG1vZHVsZXMsIGRldmljZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgbWF4X2xlbj1tYXhfbGVuLCBiYXRjaF9zaXplPWFyZ3MuYmF0Y2hfc2l6ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY2VudGVyPWNlbnRlcikKICAgIHRfZG9tID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIHQxCiAgICBwcmludChmImRvbWFpbiBwYXNzIHt0X2RvbTouMWZ9cyIpCgogICAgZGVsIG1vZGVsCiAgICBpZiBkZXZpY2UudHlwZSA9PSAiY3VkYSI6CiAgICAgICAgdG9yY2guY3VkYS5lbXB0eV9jYWNoZSgpCgogICAgdGF1cyA9IFtmbG9hdCh4KSBmb3IgeCBpbiBhcmdzLnRhdXMuc3BsaXQoIiwiKV0KICAgIHN0b3JlID0geyJtZXRhIjogeyJtb2RlbCI6IGFyZ3MubW9kZWwsICJ0YXNrIjogYXJncy50YXNrLCAibl9yZWYiOiBhcmdzLm5fcmVmLAogICAgICAgICAgICAgICAgICAgICAgIm5fZG9tIjogYXJncy5uX2RvbSwgIm1heF9sZW4iOiBtYXhfbGVuLCAicl9tYXgiOiBhcmdzLnJfbWF4LAogICAgICAgICAgICAgICAgICAgICAgImNlbnRlcmVkIjogY2VudGVyLAogICAgICAgICAgICAgICAgICAgICAgInRfcmVmX3MiOiB0X3JlZiwgInRfZG9tX3MiOiB0X2RvbSwKICAgICAgICAgICAgICAgICAgICAgICJjb3N0cyI6IHtuOiBkcmlmdF9tb2QubW9kdWxlX2Nvc3QobSkgZm9yIG4sIG0gaW4gbW9kdWxlcy5pdGVtcygpfSwKICAgICAgICAgICAgICAgICAgICAgICJkaW1zIjoge246IFttLmluX2ZlYXR1cmVzLCBtLm91dF9mZWF0dXJlc10gZm9yIG4sIG0gaW4gbW9kdWxlcy5pdGVtcygpfX0sCiAgICAgICAgICAgICAidGF1cyI6IHt9fQoKICAgIHQyID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgZm9yIHRhdSBpbiB0YXVzOgogICAgICAgIHN0b3JlWyJ0YXVzIl1bc3RyKHRhdSldID0ge30KICAgICMgTW9kdWxlLW1ham9yOiB0aGUgcmVmZXJlbmNlIGVpZ2VuZGVjb21wb3NpdGlvbiBpcyB0aGUgZXhwZW5zaXZlIHN0ZXAgYW5kIGRvZXMKICAgICMgbm90IGRlcGVuZCBvbiB0YXUsIHNvIGl0IGlzIGNvbXB1dGVkIG9uY2UgYW5kIHJldXNlZCBmb3IgZXZlcnkgdGF1LgogICAgZm9yIG5hbWUgaW4gbW9kdWxlczoKICAgICAgICAjIHByb21vdGUgb25lIG1vZHVsZSBhdCBhIHRpbWU7IHRoZSBjYWNoZWQgbWF0cmljZXMgc3RheSBmbG9hdDMyCiAgICAgICAgc2cgPSBjb3ZfZ1tuYW1lXVswXS5kb3VibGUoKQogICAgICAgIHNkID0gY292X2RbbmFtZV1bMF0uZG91YmxlKCkKICAgICAgICBldmFsc19nLCBldmVjc19nID0gZHJpZnRfbW9kLnJlZmVyZW5jZV9laWdoKHNnKQogICAgICAgIGRlbCBzZwogICAgICAgIGZvciB0YXUgaW4gdGF1czoKICAgICAgICAgICAgaWYgdGF1IDw9IDA6CiAgICAgICAgICAgICAgICB2X2NvbXAsIGsgPSBOb25lLCAwICAgICAgICAgICMgbm8gZGVmbGF0aW9uOiBwbGFpbiB0YXJnZXQgUENBCiAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICBfLCBrID0gZHJpZnRfbW9kLnN1YnNwYWNlX2Zyb21fZWlnaChldmFsc19nLCBldmVjc19nLCB0YXU9dGF1KQogICAgICAgICAgICAgICAgdl9jb21wID0gZXZlY3NfZ1s6LCBrOl0gICAgICAjIHRyYWlsaW5nIGVpZ2VudmVjdG9ycyBzcGFuIChJLVApCiAgICAgICAgICAgIGV2YWxzLCBldmVjcywgdHJfZCwgdHJfZHJpZnQgPSBkcmlmdF9tb2QuZHJpZnRfc3BlY3RydW0oCiAgICAgICAgICAgICAgICBzZCwgdl9jb21wLCByX2tlZXA9YXJncy5yX21heCwgZGV2aWNlPWRldmljZSkKICAgICAgICAgICAgc3RvcmVbInRhdXMiXVtzdHIodGF1KV1bbmFtZV0gPSB7CiAgICAgICAgICAgICAgICAiZXZhbHMiOiBldmFscy5mbG9hdCgpLm51bXB5KCksCiAgICAgICAgICAgICAgICAiYmFzaXMiOiBldmVjcy5mbG9hdCgpLm51bXB5KCksCiAgICAgICAgICAgICAgICAidHJhY2VfZCI6IHRyX2QsCiAgICAgICAgICAgICAgICAidHJhY2VfZHJpZnQiOiB0cl9kcmlmdCwKICAgICAgICAgICAgICAgICJrIjogaywKICAgICAgICAgICAgfQogICAgICAgIGRlbCBldmFsc19nLCBldmVjc19nLCBzZAogICAgICAgIGNvdl9nW25hbWVdID0gTm9uZQogICAgICAgIGNvdl9kW25hbWVdID0gTm9uZQogICAgZm9yIHRhdSBpbiB0YXVzOgogICAgICAgIHBlcl9tb2QgPSBzdG9yZVsidGF1cyJdW3N0cih0YXUpXQogICAgICAgIG1lYW5fcmF0aW8gPSBzdW0ocGVyX21vZFtuXVsidHJhY2VfZHJpZnQiXSAvIG1heChwZXJfbW9kW25dWyJ0cmFjZV9kIl0sIDFlLTEyKQogICAgICAgICAgICAgICAgICAgICAgICAgZm9yIG4gaW4gbW9kdWxlcykgLyBsZW4obW9kdWxlcykKICAgICAgICBwcmludChmInRhdT17dGF1fTogbWVhbiBkcmlmdCByYXRpbyB7bWVhbl9yYXRpbzouNGZ9IHwgIgogICAgICAgICAgICAgIGYibWVhbiBrIHtzdW0ocGVyX21vZFtuXVsnayddIGZvciBuIGluIG1vZHVsZXMpL2xlbihtb2R1bGVzKTouMWZ9IikKICAgIHRfZWlnID0gdGltZS5wZXJmX2NvdW50ZXIoKSAtIHQyCiAgICBzdG9yZVsibWV0YSJdWyJ0X2VpZ19zIl0gPSB0X2VpZwogICAgcHJpbnQoZiJzcGVjdHJhbCBhbmFseXNpcyB7dF9laWc6LjFmfXMiKQoKICAgIHRvcmNoLnNhdmUoc3RvcmUsIG91dF9wYXRoKQogICAgcHJpbnQoInNhdmVkIiwgb3V0X3BhdGgsIGYiKHtvcy5wYXRoLmdldHNpemUob3V0X3BhdGgpLzFlNjouMWZ9IE1CKSIpCgogICAgc3VtbSA9IHsia2V5Ijoga2V5LCAidF9yZWZfcyI6IHRfcmVmLCAidF9kb21fcyI6IHRfZG9tLCAidF9laWdfcyI6IHRfZWlnLAogICAgICAgICAgICAibl9tb2R1bGVzIjogbGVuKG1vZHVsZXMpLCAidGF1cyI6IHRhdXMsICJjZW50ZXJlZCI6IGNlbnRlciwKICAgICAgICAgICAgIyBtZWFuIGRyaWZ0IHJhdGlvIHBlciB0YXUsIGZvciB0aGUgcGFwZXIgdGV4dAogICAgICAgICAgICAiZHJpZnRfcmF0aW8iOiB7c3RyKHQpOiBzdW0oc3RvcmVbInRhdXMiXVtzdHIodCldW25dWyJ0cmFjZV9kcmlmdCJdCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAvIG1heChzdG9yZVsidGF1cyJdW3N0cih0KV1bbl1bInRyYWNlX2QiXSwgMWUtMTIpCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBmb3IgbiBpbiBtb2R1bGVzKSAvIGxlbihtb2R1bGVzKQogICAgICAgICAgICAgICAgICAgICAgICAgICAgZm9yIHQgaW4gdGF1c319CiAgICB3aXRoIG9wZW4ob3MucGF0aC5qb2luKFBST0ZJTEVfRElSLCBrZXkgKyAiLmpzb24iKSwgInciKSBhcyBmOgogICAgICAgIGpzb24uZHVtcChzdW1tLCBmLCBpbmRlbnQ9MikKCgppZiBfX25hbWVfXyA9PSAiX19tYWluX18iOgogICAgbWFpbigpCg==", "run.py": "IiIiU2luZ2xlLWV4cGVyaW1lbnQgcnVubmVyLgoKICAgIHB5dGhvbiBzcmMvcnVuLnB5IC0tbW9kZWwgcm9iZXJ0YS1iYXNlIC0tdGFzayBjaGVtcHJvdCAtLW1ldGhvZCBkcmlmdCAtLXNlZWQgMQoiIiIKaW1wb3J0IGFyZ3BhcnNlCmltcG9ydCBqc29uCmltcG9ydCBvcwppbXBvcnQgc3lzCmltcG9ydCB0aW1lCgppbXBvcnQgdG9yY2gKCnN5cy5wYXRoLmluc2VydCgwLCBvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSkpCmltcG9ydCBjb21tb24gICAgICAgICAgICAgICAjIG5vcWE6IEU0MDIKaW1wb3J0IGRhdGEgYXMgZGF0YV9tb2QgICAgICMgbm9xYTogRTQwMgppbXBvcnQgZW5naW5lICAgICAgICAgICAgICAgIyBub3FhOiBFNDAyCmltcG9ydCBwcm9maWxlX2RyaWZ0ICAgICAgICAjIG5vcWE6IEU0MDIKClJPT1QgPSBvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpKQpSRVNVTFRfRElSID0gb3MucGF0aC5qb2luKFJPT1QsICJydW5zIiwgInJlc3VsdHMiKQoKTkVFRFNfUFJPRklMRSA9IHsiZXZhIiwgImV2YV93aGl0ZSIsICJkcmlmdCIsICJkcmlmdF9hYnMiLCAiZHJpZnRfbm9kZWZsYXRlIn0KCgpkZWYgcnVuX2lkKGEpOgogICAgYml0cyA9IFthLm1vZGVsLnJlcGxhY2UoIi8iLCAiX18iKSwgYS50YXNrLCBhLm1ldGhvZCwgZiJye2EuYnVkZ2V0X3Jhbmt9IiwKICAgICAgICAgICAgZiJscnthLmxyOmd9IiwgZiJze2Euc2VlZH0iXQogICAgaWYgYS5tZXRob2QgaW4gTkVFRFNfUFJPRklMRToKICAgICAgICBiaXRzLmFwcGVuZChmInRhdXthLnRhdTpnfSIpCiAgICAgICAgYml0cy5hcHBlbmQoYS5zY29yZV9tb2RlKQogICAgICAgIGJpdHMuYXBwZW5kKGYiaW5pdC17YS5pbml0X21vZGV9IikKICAgICAgICBiaXRzLmFwcGVuZChmImFsbG9jLXthLmFsbG9jX21vZGV9IikKICAgICAgICBiaXRzLmFwcGVuZChmInJob3thLnJobzpnfSIpCiAgICAgICAgYml0cy5hcHBlbmQoZiJjb3Yte2EuY292fSIpCiAgICAgICAgYml0cy5hcHBlbmQoZiJzYy17YS5zY2FsZX0iKQogICAgaWYgYS50YXJnZXQgIT0gImFsbCI6CiAgICAgICAgYml0cy5hcHBlbmQoInRndC0iICsgYS50YXJnZXQpCiAgICBpZiBhLm1heF90cmFpbjoKICAgICAgICBiaXRzLmFwcGVuZChmIm57YS5tYXhfdHJhaW59IikKICAgIGlmIGEudGFnOgogICAgICAgIGJpdHMuYXBwZW5kKGEudGFnKQogICAgcmV0dXJuICJfXyIuam9pbihiaXRzKQoKCmRlZiBtYWluKCk6CiAgICBhcCA9IGFyZ3BhcnNlLkFyZ3VtZW50UGFyc2VyKCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1tb2RlbCIsIGRlZmF1bHQ9InJvYmVydGEtYmFzZSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tdGFzayIsIGRlZmF1bHQ9ImNoZW1wcm90IikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1tZXRob2QiLCBkZWZhdWx0PSJsb3JhIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zZWVkIiwgdHlwZT1pbnQsIGRlZmF1bHQ9MSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1idWRnZXRfcmFuayIsIHR5cGU9aW50LCBkZWZhdWx0PTgpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tYWxwaGEiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTE2LjApCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZHJvcG91dCIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC4wKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWxyIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0zZS00KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWVwb2NocyIsIHR5cGU9aW50LCBkZWZhdWx0PTEwKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWJhdGNoX3NpemUiLCB0eXBlPWludCwgZGVmYXVsdD0zMikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1ldmFsX2JhdGNoX3NpemUiLCB0eXBlPWludCwgZGVmYXVsdD02NCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1tYXhfbGVuIiwgdHlwZT1pbnQsIGRlZmF1bHQ9Tm9uZSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1tYXhfdHJhaW4iLCB0eXBlPWludCwgZGVmYXVsdD1Ob25lKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXdlaWdodF9kZWNheSIsIHR5cGU9ZmxvYXQsIGRlZmF1bHQ9MC4wMSkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS10YXUiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAuOTUpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcmhvIiwgdHlwZT1mbG9hdCwgZGVmYXVsdD0yLjApCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tc2NvcmVfbW9kZSIsIGRlZmF1bHQ9InJlbGF0aXZlIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1pbml0X21vZGUiLCBkZWZhdWx0PSJkcmlmdCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tYWxsb2NfbW9kZSIsIGRlZmF1bHQ9ImRyaWZ0IikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1yX21pbiIsIHR5cGU9aW50LCBkZWZhdWx0PTApCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tdGFyZ2V0IiwgZGVmYXVsdD0iYWxsIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1jb3YiLCBkZWZhdWx0PSJjZW50ZXJlZCIsIGNob2ljZXM9WyJjZW50ZXJlZCIsICJyYXciXSwKICAgICAgICAgICAgICAgICAgICBoZWxwPSJwcm9maWxlIHR5cGUgZm9yIEVWQS9EUklGVDogY292YXJpYW5jZSAoYXMgaW4gRVZBJ3MgIgogICAgICAgICAgICAgICAgICAgICAgICAgInJlZmVyZW5jZSBpbXBsZW1lbnRhdGlvbikgb3IgcmF3IHNlY29uZCBtb21lbnQiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXNjYWxlIiwgZGVmYXVsdD0iYWRqdXN0ZWQiLCBjaG9pY2VzPVsiYWRqdXN0ZWQiLCAicmFuayJdLAogICAgICAgICAgICAgICAgICAgIGhlbHA9ImFkYXB0ZXIgc2NhbGluZyBmb3IgcmFuay1yZWRpc3RyaWJ1dGluZyBtZXRob2RzOiAiCiAgICAgICAgICAgICAgICAgICAgICAgICAiJ2FkanVzdGVkJyBrZWVwcyBhbHBoYS9yIG9mIHRoZSB1bmlmb3JtIGJ1ZGdldCByYW5rIGZvciAiCiAgICAgICAgICAgICAgICAgICAgICAgICAiZXZlcnkgbW9kdWxlIChFVkEncyBkZWZhdWx0KSwgJ3JhbmsnIHVzZXMgYWxwaGEvcl9tIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1uX3JlZiIsIHR5cGU9aW50LCBkZWZhdWx0PTEwMjQpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbl9kb20iLCB0eXBlPWludCwgZGVmYXVsdD0xMDI0KQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXRhZyIsIGRlZmF1bHQ9IiIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tYW1wIiwgYWN0aW9uPSJzdG9yZV90cnVlIiwKICAgICAgICAgICAgICAgICAgICBoZWxwPSJmcDE2IGF1dG9jYXN0OyB1c2Ugb24gVHVyaW5nKyBHUFVzLCBOT1Qgb24gUGFzY2FsICIKICAgICAgICAgICAgICAgICAgICAgICAgICIoR1AxMHggcnVucyBmcDE2IGF0IDEvNjQgcmF0ZSkiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWZvcmNlIiwgYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS12ZXJib3NlIiwgYWN0aW9uPSJzdG9yZV90cnVlIikKICAgIGEgPSBhcC5wYXJzZV9hcmdzKCkKCiAgICBvcy5tYWtlZGlycyhSRVNVTFRfRElSLCBleGlzdF9vaz1UcnVlKQogICAgcmlkID0gcnVuX2lkKGEpCiAgICBvdXRfcGF0aCA9IG9zLnBhdGguam9pbihSRVNVTFRfRElSLCByaWQgKyAiLmpzb24iKQogICAgaWYgb3MucGF0aC5leGlzdHMob3V0X3BhdGgpIGFuZCBub3QgYS5mb3JjZToKICAgICAgICBwcmludCgiU0tJUCAoZXhpc3RzKToiLCByaWQpCiAgICAgICAgcmV0dXJuCgogICAgZGV2aWNlID0gdG9yY2guZGV2aWNlKCJjdWRhIiBpZiB0b3JjaC5jdWRhLmlzX2F2YWlsYWJsZSgpIGVsc2UgImNwdSIpCiAgICBjb21tb24uc2V0X3NlZWQoYS5zZWVkKQoKICAgIHRhc2sgPSBkYXRhX21vZC5sb2FkX3Rhc2soYS50YXNrLCBtYXhfdHJhaW49YS5tYXhfdHJhaW4sIHNlZWQ9MCkKICAgIG1heF9sZW4gPSBhLm1heF9sZW4gb3IgZGF0YV9tb2QuVEFTS19NQVhMRU4uZ2V0KGEudGFzaywgMTI4KQoKICAgIHByb2ZpbGUgPSBOb25lCiAgICBpZiBhLm1ldGhvZCBpbiBORUVEU19QUk9GSUxFOgogICAgICAgIGtleSA9IHByb2ZpbGVfZHJpZnQucHJvZmlsZV9rZXkoYS5tb2RlbCwgYS50YXNrLCBhLm5fcmVmLCBhLm5fZG9tLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY2VudGVyPShhLmNvdiA9PSAiY2VudGVyZWQiKSkKICAgICAgICBwID0gb3MucGF0aC5qb2luKHByb2ZpbGVfZHJpZnQuUFJPRklMRV9ESVIsIGtleSArICIucHQiKQogICAgICAgIGlmIG5vdCBvcy5wYXRoLmV4aXN0cyhwKToKICAgICAgICAgICAgcmFpc2UgU3lzdGVtRXhpdCgibWlzc2luZyBwcm9maWxlOiAiICsgcCArICJcbnJ1biBzcmMvcHJvZmlsZV9kcmlmdC5weSBmaXJzdCIpCiAgICAgICAgcHJvZmlsZSA9IHRvcmNoLmxvYWQocCwgd2VpZ2h0c19vbmx5PUZhbHNlKQoKICAgIG1vZGVsLCB0b2sgPSBlbmdpbmUuYnVpbGRfbW9kZWwoYS5tb2RlbCwgdGFza1sibnVtX2xhYmVscyJdLCB0YXNrWyJtdWx0aWxhYmVsIl0pCiAgICBpbmZvID0gZW5naW5lLmFwcGx5X21ldGhvZChtb2RlbCwgYS5tZXRob2QsIGJ1ZGdldF9yYW5rPWEuYnVkZ2V0X3JhbmssCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICBhbHBoYT1hLmFscGhhLCBkcm9wb3V0PWEuZHJvcG91dCwgcHJvZmlsZT1wcm9maWxlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgdGF1PWEudGF1LCBzY29yZV9tb2RlPWEuc2NvcmVfbW9kZSwgcmhvPWEucmhvLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgcl9taW49YS5yX21pbiwgaW5pdF9tb2RlPWEuaW5pdF9tb2RlLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgYWxsb2NfbW9kZT1hLmFsbG9jX21vZGUsCiAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICB0YXJnZXQ9YS50YXJnZXQsIHNlZWQ9YS5zZWVkLCBzY2FsZT1hLnNjYWxlKQogICAgbW9kZWwudG8oZGV2aWNlKQogICAgaWYgZGV2aWNlLnR5cGUgPT0gImN1ZGEiOgogICAgICAgIHRvcmNoLmN1ZGEucmVzZXRfcGVha19tZW1vcnlfc3RhdHMoKQoKICAgIHQwID0gdGltZS5wZXJmX2NvdW50ZXIoKQogICAgcmVzID0gZW5naW5lLnRyYWluX2V2YWwobW9kZWwsIHRvaywgdGFzaywgZGV2aWNlLCBpbmZvLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgZXBvY2hzPWEuZXBvY2hzLCBscj1hLmxyLCBiYXRjaF9zaXplPWEuYmF0Y2hfc2l6ZSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGV2YWxfYmF0Y2hfc2l6ZT1hLmV2YWxfYmF0Y2hfc2l6ZSwgbWF4X2xlbj1tYXhfbGVuLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgc2VlZD1hLnNlZWQsIHdlaWdodF9kZWNheT1hLndlaWdodF9kZWNheSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIGxvZ19ldmVyeT0xIGlmIGEudmVyYm9zZSBlbHNlIDAsIGFtcD1hLmFtcCkKICAgIHdhbGwgPSB0aW1lLnBlcmZfY291bnRlcigpIC0gdDAKCiAgICByYW5rcyA9IGluZm8uZ2V0KCJyYW5rcyIsIHt9KQogICAgcmVjb3JkID0gewogICAgICAgICJpZCI6IHJpZCwgImFyZ3MiOiB2YXJzKGEpLCAid2FsbF9zIjogd2FsbCwKICAgICAgICAibl90cmFpbiI6IGxlbih0YXNrWyJzcGxpdHMiXVsidHJhaW4iXVswXSksCiAgICAgICAgIm5fZGV2IjogbGVuKHRhc2tbInNwbGl0cyJdWyJkZXYiXVswXSksCiAgICAgICAgIm5fdGVzdCI6IGxlbih0YXNrWyJzcGxpdHMiXVsidGVzdCJdWzBdKSwKICAgICAgICAibnVtX2xhYmVscyI6IHRhc2tbIm51bV9sYWJlbHMiXSwgIm1ldHJpYyI6IHRhc2tbIm1ldHJpYyJdLAogICAgICAgICJtYXhfbGVuIjogbWF4X2xlbiwKICAgICAgICAiYnVkZ2V0Ijoge2s6IGluZm9ba10gZm9yIGsgaW4gKCJidWRnZXRfcmFuayIsICJidWRnZXRfcGFyYW1zIiwgIm5fbW9kdWxlcyIpCiAgICAgICAgICAgICAgICAgICBpZiBrIGluIGluZm99LAogICAgICAgICJzcGVudF9wYXJhbXMiOiBpbmZvLmdldCgic3BlbnRfcGFyYW1zIiksCiAgICAgICAgInJhbmtfaGlzdCI6IHtzdHIoayk6IHN1bSgxIGZvciB2IGluIHJhbmtzLnZhbHVlcygpIGlmIHYgPT0gaykKICAgICAgICAgICAgICAgICAgICAgIGZvciBrIGluIHNvcnRlZChzZXQocmFua3MudmFsdWVzKCkpKX0gaWYgcmFua3MgZWxzZSB7fSwKICAgICAgICAicmFua3MiOiB7azogaW50KHYpIGZvciBrLCB2IGluIHJhbmtzLml0ZW1zKCl9LAogICAgICAgICJyZXN1bHQiOiByZXMsCiAgICB9CiAgICBjb21tb24uc2F2ZV9qc29uKHJlY29yZCwgb3V0X3BhdGgpCiAgICBtID0gdGFza1sibWV0cmljIl0KICAgIHByaW50KGYiRE9ORSB7cmlkfVxuICB0ZXN0IHttfT17cmVzWyd0ZXN0J11bbV06LjRmfSAiCiAgICAgICAgICBmImRldj17cmVzWydkZXZfYmVzdCddLmdldChtLCBmbG9hdCgnbmFuJykpOi40Zn0gIgogICAgICAgICAgZiJhZGFwdGVyX3BhcmFtcz17cmVzWydwYXJhbXNfYWRhcHRlciddfSAiCiAgICAgICAgICBmInRpbWU9e3Jlc1sndHJhaW5fdGltZV9zJ106LjBmfXMiKQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBtYWluKCkK", "grid.py": "IiIiRXhwZXJpbWVudCBvcmNoZXN0cmF0aW9uLgoKUnVucyBhIHByaW9yaXR5LW9yZGVyZWQgbGlzdCBvZiBjb25maWd1cmF0aW9ucyBhcyBzdWJwcm9jZXNzZXMsIHNraXBwaW5nIGFueSBydW4Kd2hvc2UgcmVzdWx0IEpTT04gYWxyZWFkeSBleGlzdHMsIHNvIHRoZSB3aG9sZSBncmlkIGlzIHJlc3VtYWJsZSBhbmQgcGFydGlhbApyZXN1bHRzIGFyZSBhbHdheXMgdXNhYmxlLgoKICAgIHB5dGhvbiBzcmMvZ3JpZC5weSAtLXBsYW4gbWFpbiAtLWRyeQogICAgcHl0aG9uIHNyYy9ncmlkLnB5IC0tcGxhbiBtYWluCiIiIgppbXBvcnQgYXJncGFyc2UKaW1wb3J0IGpzb24KaW1wb3J0IG9zCmltcG9ydCBzdWJwcm9jZXNzCmltcG9ydCBzeXMKaW1wb3J0IHRpbWUKClJPT1QgPSBvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5kaXJuYW1lKG9zLnBhdGguYWJzcGF0aChfX2ZpbGVfXykpKQpfVkVOVl9QWSA9IG9zLnBhdGguam9pbihST09ULCAiLnZlbnYiLCAiU2NyaXB0cyIsICJweXRob24uZXhlIikKUFkgPSBfVkVOVl9QWSBpZiBvcy5wYXRoLmV4aXN0cyhfVkVOVl9QWSkgZWxzZSBzeXMuZXhlY3V0YWJsZQpSVU4gPSBvcy5wYXRoLmpvaW4oUk9PVCwgInNyYyIsICJydW4ucHkiKQpQUk9GID0gb3MucGF0aC5qb2luKFJPT1QsICJzcmMiLCAicHJvZmlsZV9kcmlmdC5weSIpCgpUQVNLUyA9IFsiY2hlbXByb3QiLCAicmN0MjBrIiwgImhvYyJdCk5FRURTX1BST0ZJTEUgPSB7ImV2YSIsICJldmFfd2hpdGUiLCAiZHJpZnQiLCAiZHJpZnRfYWJzIiwgImRyaWZ0X25vZGVmbGF0ZSJ9CgojIFBlci10YXNrIHRyYWluaW5nIGNvbmZpZ3VyYXRpb24uClRBU0tfQ0ZHID0gewogICAgImNoZW1wcm90IjogZGljdChlcG9jaHM9MTAsIGJhdGNoX3NpemU9MzIsIG1heF9sZW49MTI4KSwKICAgICJyY3QyMGsiOiAgIGRpY3QoZXBvY2hzPTgsICBiYXRjaF9zaXplPTMyLCBtYXhfbGVuPTk2KSwKICAgICJob2MiOiAgICAgIGRpY3QoZXBvY2hzPTEyLCBiYXRjaF9zaXplPTE2LCBtYXhfbGVuPTUxMiksCn0KCkFNUCA9IG9zLmVudmlyb24uZ2V0KCJEUklGVF9BTVAiLCAiMCIpID09ICIxIgoKIyBMZWFybmluZyByYXRlczsgZmlsbGVkIGluIGJ5IHRoZSB0dW5pbmcgcGxhbiBhbmQgdGhlbiBmcm96ZW4gaGVyZS4KTFIgPSB7CiAgICAiZnVsbCI6IDJlLTUsICJsaW5lYXIiOiAxZS0zLCAiYml0Zml0IjogMWUtMywKICAgICJsb3JhIjogM2UtNCwgImRvcmEiOiAzZS00LCAicGlzc2EiOiAzZS00LCAiYWRhbG9yYSI6IDNlLTQsCiAgICAiZXZhIjogM2UtNCwgImV2YV93aGl0ZSI6IDNlLTQsICJkcmlmdCI6IDNlLTQsICJkcmlmdF9hYnMiOiAzZS00LAogICAgImRyaWZ0X25vZGVmbGF0ZSI6IDNlLTQsCn0KCkxBRERFUiA9IFsKICAgICJnb29nbGUvYmVydF91bmNhc2VkX0wtMl9ILTEyOF9BLTIiLCAgICAjIDQuNE0gICBCRVJULVRpbnkKICAgICJnb29nbGUvYmVydF91bmNhc2VkX0wtNF9ILTI1Nl9BLTQiLCAgICAjIDExLjJNICBCRVJULU1pbmkKICAgICJnb29nbGUvYmVydF91bmNhc2VkX0wtNF9ILTUxMl9BLTgiLCAgICAjIDI4LjhNICBCRVJULVNtYWxsCiAgICAiZ29vZ2xlL2JlcnRfdW5jYXNlZF9MLThfSC01MTJfQS04IiwgICAgIyA0MS40TSAgQkVSVC1NZWRpdW0KICAgICJnb29nbGUvYmVydF91bmNhc2VkX0wtMTJfSC03NjhfQS0xMiIsICAjIDExME0gICBCRVJULUJhc2UKXQoKCmRlZiBjZmdfZm9yKHRhc2ssIG92ZXJyaWRlcz1Ob25lKToKICAgIGMgPSBkaWN0KFRBU0tfQ0ZHW3Rhc2tdKQogICAgaWYgb3ZlcnJpZGVzOgogICAgICAgIGMudXBkYXRlKG92ZXJyaWRlcykKICAgIHJldHVybiBjCgoKIyBFdmVyeSBtZXRob2QgaXMgdHVuZWQgb24gaXRzIG93biBvdmVyIHRoZSBzYW1lIGRldi1zZXQgcHJvdG9jb2wuIFZhcmlhbnRzIHVzZWQKIyBvbmx5IGluIHRoZSBhYmxhdGlvbiBpbmhlcml0IHRoZSByYXRlIHR1bmVkIGZvciB0aGVpciBwYXJlbnQgbWV0aG9kOyBhIG1ldGhvZAojIHdpdGggbm8gdHVuaW5nIHJlc3VsdHMgYXQgYWxsIChlLmcuIHRoZSBiYWNrYm9uZSBsYWRkZXIpIGZhbGxzIGJhY2sgdG8gTG9SQSdzLgpMT1JBX0ZBTUlMWSA9IHsibG9yYSIsICJkb3JhIiwgInBpc3NhIiwgImFkYWxvcmEiLCAiZXZhIiwgImV2YV93aGl0ZSIsICJkcmlmdCIsCiAgICAgICAgICAgICAgICJkcmlmdF9hYnMiLCAiZHJpZnRfbm9kZWZsYXRlIn0KUEFSRU5UID0geyJkcmlmdF9hYnMiOiAiZHJpZnQiLCAiZHJpZnRfbm9kZWZsYXRlIjogImRyaWZ0In0KX0xSX0NBQ0hFID0gTm9uZQoKCmRlZiByZXNvbHZlX2xyKG1vZGVsLCB0YXNrLCBtZXRob2QpOgogICAgIiIiQmVzdCBkZXYtc2V0IGxlYXJuaW5nIHJhdGUgZnJvbSB0aGUgdHVuaW5nIHN3ZWVwLCBlbHNlIHRoZSBkZWZhdWx0LiIiIgogICAgZ2xvYmFsIF9MUl9DQUNIRQogICAgaWYgX0xSX0NBQ0hFIGlzIE5vbmU6CiAgICAgICAgaW1wb3J0IGdsb2IKICAgICAgICBfTFJfQ0FDSEUgPSB7fQogICAgICAgIGZvciBwIGluIGdsb2IuZ2xvYihvcy5wYXRoLmpvaW4oUk9PVCwgInJ1bnMiLCAicmVzdWx0cyIsICIqdHVuZSouanNvbiIpKToKICAgICAgICAgICAgdHJ5OgogICAgICAgICAgICAgICAgd2l0aCBvcGVuKHAsIGVuY29kaW5nPSJ1dGYtOCIpIGFzIGY6CiAgICAgICAgICAgICAgICAgICAgciA9IGpzb24ubG9hZChmKQogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgYSA9IHIuZ2V0KCJhcmdzIiwge30pCiAgICAgICAgICAgIGlmIGEuZ2V0KCJ0YWciKSAhPSAidHVuZSI6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICAjIHR1bmluZyBydW5zIG9mIEVWQS9EUklGVCBtYWRlIGJlZm9yZSB0aGUgY292YXJpYW5jZS9zY2FsaW5nIGZpeAogICAgICAgICAgICAjIChubyAiY292IiBhcmcpIG11c3Qgbm90IHN0ZWVyIHRoZSBjb3JyZWN0ZWQgcnVucwogICAgICAgICAgICBpZiBhLmdldCgibWV0aG9kIikgaW4gTkVFRFNfUFJPRklMRSBhbmQgYS5nZXQoImNvdiIpICE9ICJjZW50ZXJlZCI6CiAgICAgICAgICAgICAgICBjb250aW51ZQogICAgICAgICAgICBtZXRyaWMgPSByLmdldCgibWV0cmljIiwgIm1pY3JvX2YxIikKICAgICAgICAgICAgZGV2ID0gci5nZXQoInJlc3VsdCIsIHt9KS5nZXQoImRldl9iZXN0Iiwge30pLmdldChtZXRyaWMpCiAgICAgICAgICAgIGlmIGRldiBpcyBOb25lOgogICAgICAgICAgICAgICAgY29udGludWUKICAgICAgICAgICAgayA9IChhLmdldCgibW9kZWwiKSwgYS5nZXQoInRhc2siKSwgYS5nZXQoIm1ldGhvZCIpKQogICAgICAgICAgICBpZiBrIG5vdCBpbiBfTFJfQ0FDSEUgb3IgZGV2ID4gX0xSX0NBQ0hFW2tdWzBdOgogICAgICAgICAgICAgICAgX0xSX0NBQ0hFW2tdID0gKGRldiwgYS5nZXQoImxyIikpCiAgICBvd24gPSBQQVJFTlQuZ2V0KG1ldGhvZCwgbWV0aG9kKQogICAgaGl0ID0gX0xSX0NBQ0hFLmdldCgobW9kZWwsIHRhc2ssIG93bikpCiAgICBpZiBoaXQgaXMgTm9uZSBhbmQgbWV0aG9kIGluIExPUkFfRkFNSUxZOgogICAgICAgIGhpdCA9IF9MUl9DQUNIRS5nZXQoKG1vZGVsLCB0YXNrLCAibG9yYSIpKQogICAgaWYgaGl0OgogICAgICAgIHJldHVybiBoaXRbMV0KICAgIHJldHVybiBMUlttZXRob2RdCgoKZGVmIG1ha2VfY21kKG1vZGVsLCB0YXNrLCBtZXRob2QsIHNlZWQsIGxyPU5vbmUsICoqa3cpOgogICAgYyA9IGNmZ19mb3IodGFzaywge2s6IHYgZm9yIGssIHYgaW4ga3cuaXRlbXMoKQogICAgICAgICAgICAgICAgICAgICAgIGlmIGsgaW4gKCJlcG9jaHMiLCAiYmF0Y2hfc2l6ZSIsICJtYXhfbGVuIil9KQogICAgaWYgbHIgaXMgTm9uZToKICAgICAgICBsciA9IHJlc29sdmVfbHIobW9kZWwsIHRhc2ssIG1ldGhvZCkKICAgIGNtZCA9IFtQWSwgUlVOLCAiLS1tb2RlbCIsIG1vZGVsLCAiLS10YXNrIiwgdGFzaywgIi0tbWV0aG9kIiwgbWV0aG9kLAogICAgICAgICAgICItLXNlZWQiLCBzdHIoc2VlZCksICItLWxyIiwgc3RyKGxyKSwKICAgICAgICAgICAiLS1lcG9jaHMiLCBzdHIoY1siZXBvY2hzIl0pLCAiLS1iYXRjaF9zaXplIiwgc3RyKGNbImJhdGNoX3NpemUiXSksCiAgICAgICAgICAgIi0tbWF4X2xlbiIsIHN0cihjWyJtYXhfbGVuIl0pXQogICAgZm9yIGsgaW4gKCJidWRnZXRfcmFuayIsICJ0YXUiLCAicmhvIiwgInNjb3JlX21vZGUiLCAiaW5pdF9tb2RlIiwKICAgICAgICAgICAgICAiYWxsb2NfbW9kZSIsICJ0YXJnZXQiLCAibWF4X3RyYWluIiwgInRhZyIpOgogICAgICAgIGlmIGsgaW4ga3cgYW5kIGt3W2tdIGlzIG5vdCBOb25lOgogICAgICAgICAgICBjbWQgKz0gWyItLSIgKyBrLCBzdHIoa3dba10pXQogICAgaWYgQU1QOgogICAgICAgIGNtZCArPSBbIi0tYW1wIl0KICAgIHJldHVybiBjbWQKCgojIC0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tCiMgcGxhbnMKIyAtLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLS0tLQpkZWYgcGxhbl90dW5lKG1vZGVsKToKICAgICIiIkxSIHNlbGVjdGlvbiwgb25lIHNlZWQsIGRldi1zZXQgZGVjaXNpb24sIGZvciBldmVyeSBtYWluLXRhYmxlIG1ldGhvZC4iIiIKICAgIG91dCA9IFtdCiAgICAjIEVhY2ggYmFzZWxpbmUgZ3JpZCBleHRlbmRzIG9uZSBzdGVwIHBhc3QgdGhlIHZhbHVlIGZpcnN0IHNlbGVjdGVkLCBzbyBubwogICAgIyBiYXNlbGluZSdzIGNob3NlbiByYXRlIHNpdHMgb24gdGhlIGVkZ2Ugb2YgaXRzIHNlYXJjaCByYW5nZS4KICAgIGdyaWRzID0geyJsb3JhIjogWzFlLTQsIDNlLTQsIDFlLTNdLCAiZnVsbCI6IFsxZS01LCAzZS01LCA1ZS01XSwKICAgICAgICAgICAgICJsaW5lYXIiOiBbMWUtMywgNWUtMywgMmUtMl0sICJiaXRmaXQiOiBbM2UtNCwgMWUtMywgM2UtM119CiAgICAjIGV2ZXJ5IG90aGVyIGxvdy1yYW5rIG1ldGhvZCBnZXRzIGl0cyBvd24gc2VhcmNoOyAxZS0zIGlzIG9taXR0ZWQgYmVjYXVzZQogICAgIyBpdCBhbHJlYWR5IGRpdmVyZ2VzIGZvciBwbGFpbiBMb1JBIG9uIEhvQwogICAgZm9yIG0gaW4gKCJkb3JhIiwgInBpc3NhIiwgImFkYWxvcmEiKToKICAgICAgICBncmlkc1ttXSA9IFsxZS00LCAzZS00XQogICAgIyBwcm9maWxlLWluaXRpYWxpc2VkIG1ldGhvZHMgZ2V0IHR3byBsb3dlciByYXRlcyBhcyB3ZWxsOiBFVkEncyBwcmluY2lwYWwKICAgICMgZGlyZWN0aW9ucyBjYXJyeSBSb0JFUlRhJ3MgaGlnaC12YXJpYW5jZSBvdXRsaWVyIGZlYXR1cmVzIGFuZCBkaXZlcmdlIGF0CiAgICAjIDFlLTQgYW5kIGFib3ZlLCBzbyBpdHMgZ3JpZCBtdXN0IHJlYWNoIGRvd24gdG8gd2hlcmUgaXQgY2FuIHRyYWluLiBEUklGVAogICAgIyBnZXRzIHRoZSBpZGVudGljYWwgZ3JpZCBzbyB0aGUgY29tcGFyaXNvbiBzdGF5cyBtYXRjaGVkLgogICAgZm9yIG0gaW4gKCJldmEiLCAiZXZhX3doaXRlIiwgImRyaWZ0Iik6CiAgICAgICAgZ3JpZHNbbV0gPSBbMWUtNSwgM2UtNSwgMWUtNCwgM2UtNF0KICAgIGZvciB0YXNrIGluIFRBU0tTOgogICAgICAgIGZvciBtZXRob2QsIGxycyBpbiBncmlkcy5pdGVtcygpOgogICAgICAgICAgICBmb3IgbHIgaW4gbHJzOgogICAgICAgICAgICAgICAgb3V0LmFwcGVuZChtYWtlX2NtZChtb2RlbCwgdGFzaywgbWV0aG9kLCAxLCBscj1sciwgdGFnPSJ0dW5lIikpCiAgICByZXR1cm4gb3V0CgoKZGVmIHBsYW5fbWFpbihtb2RlbCwgc2VlZHM9KDEsIDIsIDMpLCBtZXRob2RzPU5vbmUpOgogICAgbWV0aG9kcyA9IG1ldGhvZHMgb3IgWyJkcmlmdCIsICJsb3JhIiwgImV2YSIsICJldmFfd2hpdGUiLCAiYWRhbG9yYSIsICJmdWxsIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAiYml0Zml0IiwgImxpbmVhciIsICJwaXNzYSIsICJkb3JhIl0KICAgIG91dCA9IFtdCiAgICAjIHNlZWQtbWFqb3Igb3JkZXJpbmc6IGEgY29tcGxldGUgMS1zZWVkIHRhYmxlIGV4aXN0cyBhcyBlYXJseSBhcyBwb3NzaWJsZQogICAgZm9yIHNlZWQgaW4gc2VlZHM6CiAgICAgICAgZm9yIHRhc2sgaW4gVEFTS1M6CiAgICAgICAgICAgIGZvciBtZXRob2QgaW4gbWV0aG9kczoKICAgICAgICAgICAgICAgIG91dC5hcHBlbmQobWFrZV9jbWQobW9kZWwsIHRhc2ssIG1ldGhvZCwgc2VlZCkpCiAgICByZXR1cm4gb3V0CgoKU1dFRVBfTUVUSE9EUyA9IFsiZHJpZnQiLCAibG9yYSIsICJldmEiLCAiZXZhX3doaXRlIl0KCgpkZWYgcGxhbl9sYWRkZXIobW9kZWxfbGlzdD1Ob25lLCBzZWVkcz0oMSwgMiwgMyksIHRhc2s9ImNoZW1wcm90IiwKICAgICAgICAgICAgICAgIGxyX2Zyb209InJvYmVydGEtYmFzZSIpOgogICAgIiIiVGhlIHNtYWxsIGJhY2tib25lcyBhcmUgbm90IHR1bmVkIHNlcGFyYXRlbHk6IGVhY2ggbWV0aG9kIHVzZXMgdGhlIHJhdGUKICAgIGl0IHdhcyB0dW5lZCB0byBvbiB0aGUgcHJpbWFyeSBiYWNrYm9uZSBmb3IgdGhlIHNhbWUgdGFzay4iIiIKICAgIG1vZGVscyA9IG1vZGVsX2xpc3Qgb3IgTEFEREVSCiAgICAjIERSSUZUX0xBRERFUl9MUiAoSlNPTiB7bWV0aG9kOiBscn0pIHBpbnMgdGhlIHJhdGVzIHdoZW4gdGhlIHR1bmluZyByZXN1bHRzCiAgICAjIGFyZSBub3QgYXZhaWxhYmxlIGluIHRoaXMgc2Vzc2lvbiwgZS5nLiBhIGxhZGRlci1vbmx5IHJ1bgogICAgcGlubmVkID0ganNvbi5sb2Fkcyhvcy5lbnZpcm9uLmdldCgiRFJJRlRfTEFEREVSX0xSIiwgInt9IikpCiAgICBvdXQgPSBbXQogICAgZm9yIHNlZWQgaW4gc2VlZHM6CiAgICAgICAgZm9yIG1vZGVsIGluIG1vZGVsczoKICAgICAgICAgICAgZm9yIG1ldGhvZCBpbiBTV0VFUF9NRVRIT0RTOgogICAgICAgICAgICAgICAgbHIgPSBwaW5uZWQuZ2V0KG1ldGhvZCkgb3IgcmVzb2x2ZV9scihscl9mcm9tLCB0YXNrLCBtZXRob2QpCiAgICAgICAgICAgICAgICBvdXQuYXBwZW5kKG1ha2VfY21kKG1vZGVsLCB0YXNrLCBtZXRob2QsIHNlZWQsIGxyPWxyKSkKICAgIHJldHVybiBvdXQKCgpkZWYgcGxhbl9sYWRkZXJfbHIobW9kZWxfbGlzdD1Ob25lLCBzZWVkcz0oMSwgMiwgMyksIHRhc2s9ImNoZW1wcm90IiwKICAgICAgICAgICAgICAgICAgIGxyX2Zyb209InJvYmVydGEtYmFzZSIpOgogICAgIiIiQ29udHJvbCBmb3IgdGhlIGxhZGRlcjogRVZBIGF0IHRoZSBsZWFybmluZyByYXRlIHRoZSBvdGhlciBsb3ctcmFuawogICAgbWV0aG9kcyB1c2UgdGhlcmUgKExvUkEncyksIGluc3RlYWQgb2YgdGhlIGxvd2VyIHJhdGUgRVZBIHdhcyB0dW5lZCB0byBvbiB0aGUKICAgIHByaW1hcnkgYmFja2JvbmUuIFRhZ2dlZCBzbyBpdCBuZXZlciByZXBsYWNlcyB0aGUgdHVuZWQtcmF0ZSBydW5zIGluIHRoZQogICAgdGFibGVzLiIiIgogICAgbW9kZWxzID0gbW9kZWxfbGlzdCBvciBMQURERVIKICAgIHBpbm5lZCA9IGpzb24ubG9hZHMob3MuZW52aXJvbi5nZXQoIkRSSUZUX0xBRERFUl9MUiIsICJ7fSIpKQogICAgbHIgPSBwaW5uZWQuZ2V0KCJsb3JhIikgb3IgcmVzb2x2ZV9scihscl9mcm9tLCB0YXNrLCAibG9yYSIpCiAgICByZXR1cm4gW21ha2VfY21kKG1vZGVsLCB0YXNrLCAiZXZhIiwgc2VlZCwgbHI9bHIsIHRhZz0ibGFkZGVyTFIiKQogICAgICAgICAgICBmb3Igc2VlZCBpbiBzZWVkcyBmb3IgbW9kZWwgaW4gbW9kZWxzXQoKCmRlZiBwaW5uZWRfbHIobW9kZWwsIHRhc2ssIG1ldGhvZCk6CiAgICAiIiJUaGUgdHVuZWQgcmF0ZSwgcGlubmVkIGZyb20gdGhlIGxvY2FsIHJlc3VsdHMgd2hlbiB0aGUgdHVuaW5nIHJ1bnMgYXJlIG5vdAogICAgcmVzdG9yYWJsZSBpbiBhIHJlbW90ZSBzZXNzaW9uIChEUklGVF9QSU5ORURfTFIsIEpTT04ge3Rhc2s6IHttZXRob2Q6IGxyfX0pLiIiIgogICAgcGlubmVkID0ganNvbi5sb2Fkcyhvcy5lbnZpcm9uLmdldCgiRFJJRlRfUElOTkVEX0xSIiwgInt9IikpCiAgICByZXR1cm4gcGlubmVkLmdldCh0YXNrLCB7fSkuZ2V0KFBBUkVOVC5nZXQobWV0aG9kLCBtZXRob2QpKSBvciBcCiAgICAgICAgcmVzb2x2ZV9scihtb2RlbCwgdGFzaywgbWV0aG9kKQoKCmRlZiBwbGFuX3BsYWNlbWVudF94KG1vZGVsLCBzZWVkcz0oMSwgMiwgMyksIHRhc2tzPSgicmN0MjBrIiwgImhvYyIpKToKICAgICIiIlRoZSBhZGFwdGVyLXBsYWNlbWVudCBhYmxhdGlvbiBvZiBwbGFuX2FibGF0aW9uLCBvbiB0aGUgb3RoZXIgdHdvIHRhc2tzLiIiIgogICAgcmV0dXJuIFttYWtlX2NtZChtb2RlbCwgdGFzaywgbWV0aG9kLCBzZWVkLCBscj1waW5uZWRfbHIobW9kZWwsIHRhc2ssIG1ldGhvZCksCiAgICAgICAgICAgICAgICAgICAgIHRhcmdldD10Z3QpCiAgICAgICAgICAgIGZvciBzZWVkIGluIHNlZWRzIGZvciB0YXNrIGluIHRhc2tzCiAgICAgICAgICAgIGZvciBtZXRob2QgaW4gKCJsb3JhIiwgImRyaWZ0IikgZm9yIHRndCBpbiAoImF0dG4iLCAiZmZuIildCgoKZGVmIHBsYW5fc2VlZHM0NShtb2RlbCwgc2VlZHM9KDQsIDUpKToKICAgICIiIlR3byBmdXJ0aGVyIHNlZWRzIGZvciB0aGUgZm91ciBtZXRob2RzIHRoZSBhbmFseXNpcyB0dXJucyBvbi4gVGFnZ2VkLCBzbyB0aGUKICAgIG1haW4gdGFibGUga2VlcHMgdGhlIHRocmVlIHNlZWRzIGV2ZXJ5IG1ldGhvZCBoYXMuIiIiCiAgICByZXR1cm4gW21ha2VfY21kKG1vZGVsLCB0YXNrLCBtZXRob2QsIHNlZWQsIGxyPXBpbm5lZF9scihtb2RlbCwgdGFzaywgbWV0aG9kKSwKICAgICAgICAgICAgICAgICAgICAgdGFnPSJzZWVkczQ1IikKICAgICAgICAgICAgZm9yIHNlZWQgaW4gc2VlZHMgZm9yIHRhc2sgaW4gVEFTS1MKICAgICAgICAgICAgZm9yIG1ldGhvZCBpbiAoImxvcmEiLCAiZXZhIiwgImV2YV93aGl0ZSIsICJkcmlmdCIpXQoKCkRFQ09ERVIgPSAiSHVnZ2luZ0ZhY2VUQi9TbW9sTE0yLTM2ME0iCkRFQ09ERVJfTUVUSE9EUyA9ICgibG9yYSIsICJldmEiLCAiZXZhX3doaXRlIiwgImRyaWZ0IikKCgpkZWYgcGxhbl9kZWNvZGVyX3R1bmUobW9kZWwsIHRhc2s9ImNoZW1wcm90Iik6CiAgICAiIiJMZWFybmluZyByYXRlcyBmb3IgdGhlIGRlY29kZXIgU0xNLCB0dW5lZCBvbiBpdHMgb3duIGRldiBzZXQgd2l0aCB0aGUgZ3JpZHMKICAgIHVzZWQgZm9yIFJvQkVSVGEtYmFzZSAocHJvZmlsZS1pbml0aWFsaXNlZCBtZXRob2RzIHJlYWNoIHR3byByYXRlcyBsb3dlcikuIiIiCiAgICBncmlkcyA9IHsibG9yYSI6IFsxZS00LCAzZS00LCAxZS0zXX0KICAgIGZvciBtIGluICgiZXZhIiwgImV2YV93aGl0ZSIsICJkcmlmdCIpOgogICAgICAgIGdyaWRzW21dID0gWzFlLTUsIDNlLTUsIDFlLTQsIDNlLTRdCiAgICByZXR1cm4gW21ha2VfY21kKG1vZGVsLCB0YXNrLCBtLCAxLCBscj1sciwgdGFnPSJ0dW5lIikKICAgICAgICAgICAgZm9yIG0sIGxycyBpbiBncmlkcy5pdGVtcygpIGZvciBsciBpbiBscnNdCgoKZGVmIHBsYW5fZGVjb2Rlcl90dW5lX2V4dChtb2RlbCwgdGFzaz0iY2hlbXByb3QiKToKICAgICIiIkV2ZXJ5IG1ldGhvZCBjaG9zZSB0aGUgdG9wIG9mIGl0cyBmaXJzdCBncmlkIG9uIHRoZSBkZWNvZGVyLCBzbyBlYWNoIGdyaWQKICAgIGV4dGVuZHMgcGFzdCBpdCAoYW5kIHRoZSBwcm9maWxlLWluaXRpYWxpc2VkIG1ldGhvZHMgbm93IHJlYWNoIExvUkEncyByYXRlKS4iIiIKICAgIGdyaWRzID0geyJsb3JhIjogWzNlLTNdfQogICAgZm9yIG0gaW4gKCJldmEiLCAiZXZhX3doaXRlIiwgImRyaWZ0Iik6CiAgICAgICAgZ3JpZHNbbV0gPSBbMWUtMywgM2UtM10KICAgIHJldHVybiBbbWFrZV9jbWQobW9kZWwsIHRhc2ssIG0sIDEsIGxyPWxyLCB0YWc9InR1bmUiKQogICAgICAgICAgICBmb3IgbSwgbHJzIGluIGdyaWRzLml0ZW1zKCkgZm9yIGxyIGluIGxyc10KCgpkZWYgcGxhbl9kZWNvZGVyX21haW4obW9kZWwsIHNlZWRzPSgxLCAyLCAzKSwgdGFzaz0iY2hlbXByb3QiKToKICAgIHJldHVybiBbbWFrZV9jbWQobW9kZWwsIHRhc2ssIG0sIHNlZWQpIGZvciBzZWVkIGluIHNlZWRzIGZvciBtIGluIERFQ09ERVJfTUVUSE9EU10KCgpkZWYgcGxhbl9idWRnZXQobW9kZWwsIHNlZWRzPSgxLCAyKSwgdGFzaz0iY2hlbXByb3QiKToKICAgIG91dCA9IFtdCiAgICBmb3Igc2VlZCBpbiBzZWVkczoKICAgICAgICBmb3IgciBpbiBbMSwgMiwgNCwgOCwgMTZdOgogICAgICAgICAgICBmb3IgbWV0aG9kIGluIFNXRUVQX01FVEhPRFM6CiAgICAgICAgICAgICAgICBvdXQuYXBwZW5kKG1ha2VfY21kKG1vZGVsLCB0YXNrLCBtZXRob2QsIHNlZWQsIGJ1ZGdldF9yYW5rPXIpKQogICAgcmV0dXJuIG91dAoKCmRlZiBwbGFuX2FibGF0aW9uKG1vZGVsLCBzZWVkcz0oMSwgMiwgMyksIHRhc2s9ImNoZW1wcm90Iik6CiAgICBvdXQgPSBbXQogICAgZm9yIHNlZWQgaW4gc2VlZHM6CiAgICAgICAgIyB0YXUgc3dlZXAKICAgICAgICBmb3IgdGF1IGluIFswLjAsIDAuNSwgMC45LCAwLjk1LCAwLjk5XToKICAgICAgICAgICAgb3V0LmFwcGVuZChtYWtlX2NtZChtb2RlbCwgdGFzaywgImRyaWZ0Iiwgc2VlZCwgdGF1PXRhdSkpCiAgICAgICAgIyBmYWN0b3Jpc2VkOiBhbGxvY2F0aW9uIHZzIGluaXRpYWxpc2F0aW9uCiAgICAgICAgb3V0LmFwcGVuZChtYWtlX2NtZChtb2RlbCwgdGFzaywgImRyaWZ0Iiwgc2VlZCwgaW5pdF9tb2RlPSJyYW5kb20iLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgdGFnPSJhbGxvY09ubHkiKSkKICAgICAgICBvdXQuYXBwZW5kKG1ha2VfY21kKG1vZGVsLCB0YXNrLCAiZHJpZnQiLCBzZWVkLCBhbGxvY19tb2RlPSJ1bmlmb3JtIiwKICAgICAgICAgICAgICAgICAgICAgICAgICAgIHRhZz0iaW5pdE9ubHkiKSkKICAgICAgICBvdXQuYXBwZW5kKG1ha2VfY21kKG1vZGVsLCB0YXNrLCAiZHJpZnQiLCBzZWVkLCBpbml0X21vZGU9InJhbmRfb3J0aG8iLAogICAgICAgICAgICAgICAgICAgICAgICAgICAgdGFnPSJyYW5kT3J0aG8iKSkKICAgICAgICAjIHNjb3JpbmcgdmFyaWFudAogICAgICAgIG91dC5hcHBlbmQobWFrZV9jbWQobW9kZWwsIHRhc2ssICJkcmlmdF9hYnMiLCBzZWVkKSkKICAgICAgICAjIG1vZHVsZSB0YXJnZXRpbmcKICAgICAgICBmb3IgdGd0IGluIFsiYXR0biIsICJmZm4iXToKICAgICAgICAgICAgb3V0LmFwcGVuZChtYWtlX2NtZChtb2RlbCwgdGFzaywgImRyaWZ0Iiwgc2VlZCwgdGFyZ2V0PXRndCkpCiAgICAgICAgICAgIG91dC5hcHBlbmQobWFrZV9jbWQobW9kZWwsIHRhc2ssICJsb3JhIiwgc2VlZCwgdGFyZ2V0PXRndCkpCiAgICByZXR1cm4gb3V0CgoKUExBTlMgPSB7InR1bmUiOiBwbGFuX3R1bmUsICJtYWluIjogcGxhbl9tYWluLCAibGFkZGVyIjogcGxhbl9sYWRkZXIsCiAgICAgICAgICJsYWRkZXJfbHIiOiBwbGFuX2xhZGRlcl9sciwgImJ1ZGdldCI6IHBsYW5fYnVkZ2V0LAogICAgICAgICAiYWJsYXRpb24iOiBwbGFuX2FibGF0aW9uLCAicGxhY2VtZW50X3giOiBwbGFuX3BsYWNlbWVudF94LAogICAgICAgICAic2VlZHM0NSI6IHBsYW5fc2VlZHM0NSwgImRlY29kZXJfdHVuZSI6IHBsYW5fZGVjb2Rlcl90dW5lLAogICAgICAgICAiZGVjb2Rlcl90dW5lX2V4dCI6IHBsYW5fZGVjb2Rlcl90dW5lX2V4dCwgImRlY29kZXJfbWFpbiI6IHBsYW5fZGVjb2Rlcl9tYWlufQoKCiMgV2FsbC1jbG9jayBjYXBzIG9uIGV2ZXJ5IGNoaWxkIHByb2Nlc3MuIEEgc3RhbGxlZCBjaGVja3BvaW50IGRvd25sb2FkIG9uY2UgaHVuZwojIGEgcHJvZmlsaW5nIHN0ZXAgZm9yIDUuNiBoIHVudGlsIHRoZSBwbGF0Zm9ybSBraWxsZWQgdGhlIHNlc3Npb247IHdpdGggYSBjYXAgdGhlCiMgc3RlcCBmYWlscywgdGhlIHJ1biB0aGF0IG5lZWRlZCBpdCBmYWlscyBmYXN0LCBhbmQgZXZlcnl0aGluZyBlbHNlIHByb2NlZWRzLgpQUk9GSUxFX1RJTUVPVVRfUyA9IDMwICogNjAKUlVOX1RJTUVPVVRfUyA9IDYwICogNjAKCgpkZWYgX3J1bl9jYXBwZWQoY21kLCB0aW1lb3V0KToKICAgIHRyeToKICAgICAgICByZXR1cm4gc3VicHJvY2Vzcy5ydW4oY21kLCB0aW1lb3V0PXRpbWVvdXQpLnJldHVybmNvZGUKICAgIGV4Y2VwdCBzdWJwcm9jZXNzLlRpbWVvdXRFeHBpcmVkOgogICAgICAgIHByaW50KGYiICAhISB0aW1lb3V0IGFmdGVyIHt0aW1lb3V0LzYwOi4wZn0gbWluOiB7JyAnLmpvaW4oY21kWzI6OF0pfSIsCiAgICAgICAgICAgICAgZmx1c2g9VHJ1ZSkKICAgICAgICByZXR1cm4gLTkKCgpkZWYgZW5zdXJlX3Byb2ZpbGVzKGNtZHMsIGRlYWRsaW5lPTApOgogICAgIiIiQ29tcHV0ZSBhbnkgRFJJRlQgcHJvZmlsZSBhIHF1ZXVlZCBydW4gd2lsbCBuZWVkLiIiIgogICAgbmVlZCA9IHNldCgpCiAgICBmb3IgYyBpbiBjbWRzOgogICAgICAgIGQgPSBkaWN0KHppcChjWzI6OjJdLCBjWzM6OjJdKSkgICAgICMgZmxhZ3Mgc3RhcnQgYXQgaW5kZXggMgogICAgICAgIG1ldGhvZCA9IGQuZ2V0KCItLW1ldGhvZCIpCiAgICAgICAgaWYgbWV0aG9kIGluIE5FRURTX1BST0ZJTEU6CiAgICAgICAgICAgIG5lZWQuYWRkKChkWyItLW1vZGVsIl0sIGRbIi0tdGFzayJdKSkKICAgIGZvciBtb2RlbCwgdGFzayBpbiBzb3J0ZWQobmVlZCk6CiAgICAgICAgaWYgZGVhZGxpbmUgYW5kIHRpbWUudGltZSgpID4gZGVhZGxpbmU6CiAgICAgICAgICAgIHByaW50KCJERUFETElORSByZWFjaGVkOiBza2lwcGluZyByZW1haW5pbmcgcHJvZmlsZXMiLCBmbHVzaD1UcnVlKQogICAgICAgICAgICByZXR1cm4KICAgICAgICBwcmludChmIltwcm9maWxlXSB7bW9kZWx9IC8ge3Rhc2t9IiwgZmx1c2g9VHJ1ZSkKICAgICAgICBfcnVuX2NhcHBlZChbUFksIFBST0YsICItLW1vZGVsIiwgbW9kZWwsICItLXRhc2siLCB0YXNrXSwgUFJPRklMRV9USU1FT1VUX1MpCgoKZGVmIG1haW4oKToKICAgIGFwID0gYXJncGFyc2UuQXJndW1lbnRQYXJzZXIoKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLXBsYW4iLCByZXF1aXJlZD1UcnVlLCBjaG9pY2VzPWxpc3QoUExBTlMpKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLW1vZGVsIiwgZGVmYXVsdD0icm9iZXJ0YS1iYXNlIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zZWVkcyIsIGRlZmF1bHQ9IjEsMiwzIikKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1kcnkiLCBhY3Rpb249InN0b3JlX3RydWUiKQogICAgYXAuYWRkX2FyZ3VtZW50KCItLWxpbWl0IiwgdHlwZT1pbnQsIGRlZmF1bHQ9MCkKICAgIGFwLmFkZF9hcmd1bWVudCgiLS1zaGFyZCIsIGRlZmF1bHQ9IjAvMSIsCiAgICAgICAgICAgICAgICAgICAgaGVscD0iay9uOiBydW4gZXZlcnkgbi10aCBjb21tYW5kIHN0YXJ0aW5nIGF0IGsgKG9uZSBwZXIgR1BVKSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tZGVhZGxpbmUiLCB0eXBlPWZsb2F0LCBkZWZhdWx0PTAsCiAgICAgICAgICAgICAgICAgICAgaGVscD0idW5peCB0aW1lIGFmdGVyIHdoaWNoIG5vIG5ldyBydW4gaXMgc3RhcnRlZCIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tcHJvZmlsZXNfb25seSIsIGFjdGlvbj0ic3RvcmVfdHJ1ZSIpCiAgICBhcC5hZGRfYXJndW1lbnQoIi0tbm9fcHJvZmlsZXMiLCBhY3Rpb249InN0b3JlX3RydWUiKQogICAgYSA9IGFwLnBhcnNlX2FyZ3MoKQoKICAgIHNlZWRzID0gdHVwbGUoaW50KHMpIGZvciBzIGluIGEuc2VlZHMuc3BsaXQoIiwiKSkKICAgIGZuID0gUExBTlNbYS5wbGFuXQogICAgaWYgYS5wbGFuIGluICgidHVuZSIsICJkZWNvZGVyX3R1bmUiLCAiZGVjb2Rlcl90dW5lX2V4dCIpOgogICAgICAgIGNtZHMgPSBmbihhLm1vZGVsKQogICAgZWxpZiBhLnBsYW4gaW4gKCJsYWRkZXIiLCAibGFkZGVyX2xyIik6CiAgICAgICAgY21kcyA9IGZuKHNlZWRzPXNlZWRzKQogICAgZWxzZToKICAgICAgICBjbWRzID0gZm4oYS5tb2RlbCwgc2VlZHM9c2VlZHMpCiAgICBpZiBhLmxpbWl0OgogICAgICAgIGNtZHMgPSBjbWRzWzphLmxpbWl0XQogICAgaywgbiA9IChpbnQoeCkgZm9yIHggaW4gYS5zaGFyZC5zcGxpdCgiLyIpKQogICAgY21kcyA9IGNtZHNbazo6bl0KCiAgICBwcmludChmInBsYW49e2EucGxhbn0gbW9kZWw9e2EubW9kZWx9IHNoYXJkPXtrfS97bn0gcnVucz17bGVuKGNtZHMpfSIpCiAgICBpZiBhLmRyeToKICAgICAgICBmb3IgYyBpbiBjbWRzWzoyMDBdOgogICAgICAgICAgICBwcmludCgiICIsICIgIi5qb2luKGNbMjpdKSkKICAgICAgICByZXR1cm4KCiAgICBpZiBub3QgYS5ub19wcm9maWxlczoKICAgICAgICBlbnN1cmVfcHJvZmlsZXMoY21kcywgYS5kZWFkbGluZSkKICAgIGlmIGEucHJvZmlsZXNfb25seToKICAgICAgICByZXR1cm4KCiAgICB0MCA9IHRpbWUucGVyZl9jb3VudGVyKCkKICAgIGRvbmUgPSAwCiAgICBmb3IgaSwgYyBpbiBlbnVtZXJhdGUoY21kcywgMSk6CiAgICAgICAgaWYgYS5kZWFkbGluZSBhbmQgdGltZS50aW1lKCkgPiBhLmRlYWRsaW5lOgogICAgICAgICAgICBwcmludChmIlxuREVBRExJTkUgcmVhY2hlZDogc3RvcHBpbmcgYmVmb3JlIHJ1biB7aX0ve2xlbihjbWRzKX07ICIKICAgICAgICAgICAgICAgICAgInJlLXJ1biBuZXh0IHNlc3Npb24gdG8gY29udGludWUiLCBmbHVzaD1UcnVlKQogICAgICAgICAgICByZXR1cm4KICAgICAgICBwcmludChmIlxuW3tpfS97bGVuKGNtZHMpfV0geycgJy5qb2luKGNbMzpdKX0iLCBmbHVzaD1UcnVlKQogICAgICAgIHJjID0gX3J1bl9jYXBwZWQoYywgUlVOX1RJTUVPVVRfUykKICAgICAgICBpZiByYyAhPSAwOgogICAgICAgICAgICBwcmludChmIiAgISEgZXhpdCB7cmN9IiwgZmx1c2g9VHJ1ZSkKICAgICAgICBkb25lICs9IDEKICAgICAgICBlbCA9IHRpbWUucGVyZl9jb3VudGVyKCkgLSB0MAogICAgICAgIHByaW50KGYiICBlbGFwc2VkIHtlbC82MDouMWZ9IG1pbiB8IGF2ZyB7ZWwvZG9uZS82MDouMmZ9IG1pbi9ydW4gfCAiCiAgICAgICAgICAgICAgZiJldGEgeyhsZW4oY21kcyktZG9uZSkqZWwvZG9uZS8zNjAwOi4yZn0gaCIsIGZsdXNoPVRydWUpCiAgICBwcmludCgiR1JJRCBDT01QTEVURSIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo=", "analyze.py": "...", "download_data.py": "IiIiRmV0Y2ggZXZlcnkgZGF0YXNldCB1c2VkIGluIHRoZSBwYXBlci4gSWRlbXBvdGVudDsgc2FmZSB0byByZS1ydW4uIiIiCmltcG9ydCBvcwppbXBvcnQgc3lzCmltcG9ydCB0aW1lCmltcG9ydCB1cmxsaWIucmVxdWVzdAoKUk9PVCA9IG9zLnBhdGguZGlybmFtZShvcy5wYXRoLmRpcm5hbWUob3MucGF0aC5hYnNwYXRoKF9fZmlsZV9fKSkpCkRBVEEgPSBvcy5wYXRoLmpvaW4oUk9PVCwgImRhdGEiKQoKUzMgPSAiaHR0cHM6Ly9hbGxlbm5scC5zMy11cy13ZXN0LTIuYW1hem9uYXdzLmNvbS9kb250X3N0b3BfcHJldHJhaW5pbmcvZGF0YSIKSEYgPSAiaHR0cHM6Ly9odWdnaW5nZmFjZS5jby9hcGkvZGF0YXNldHMiCgpGSUxFUyA9IFsKICAgICMgQ2hlbVByb3Q6IDEzLXdheSBjaGVtaWNhbC1wcm90ZWluIHJlbGF0aW9uIGNsYXNzaWZpY2F0aW9uIChCaW9DcmVhdGl2ZSBWSSksCiAgICAjIGluIHRoZSBzcGxpdCByZWxlYXNlZCB3aXRoIEd1cnVyYW5nYW4gZXQgYWwuICgyMDIwKS4KICAgIChmIntTM30vY2hlbXByb3QvdHJhaW4uanNvbmwiLCAiY2hlbXByb3QvdHJhaW4uanNvbmwiKSwKICAgIChmIntTM30vY2hlbXByb3QvZGV2Lmpzb25sIiwgImNoZW1wcm90L2Rldi5qc29ubCIpLAogICAgKGYie1MzfS9jaGVtcHJvdC90ZXN0Lmpzb25sIiwgImNoZW1wcm90L3Rlc3QuanNvbmwiKSwKICAgICMgUkNULTIwazogNS13YXkgc2VudGVuY2Utcm9sZSBjbGFzc2lmaWNhdGlvbiBpbiBSQ1QgYWJzdHJhY3RzLgogICAgKGYie1MzfS9yY3QtMjBrL3RyYWluLmpzb25sIiwgInJjdDIway90cmFpbi5qc29ubCIpLAogICAgKGYie1MzfS9yY3QtMjBrL2Rldi5qc29ubCIsICJyY3QyMGsvZGV2Lmpzb25sIiksCiAgICAoZiJ7UzN9L3JjdC0yMGsvdGVzdC5qc29ubCIsICJyY3QyMGsvdGVzdC5qc29ubCIpLAogICAgIyBIYWxsbWFya3Mgb2YgQ2FuY2VyLCBzZW50ZW5jZS1sZXZlbCByZWxlYXNlIChhZ2dyZWdhdGVkIHRvIGRvY3VtZW50cyBpbiBkYXRhLnB5KS4KICAgIChmIntIRn0vcWFuYXN0ZWsvSG9DL3BhcnF1ZXQvSG9DL3RyYWluLzAucGFycXVldCIsICJob2MvdHJhaW4ucGFycXVldCIpLAogICAgKGYie0hGfS9xYW5hc3Rlay9Ib0MvcGFycXVldC9Ib0MvdmFsaWRhdGlvbi8wLnBhcnF1ZXQiLCAiaG9jL3ZhbGlkYXRpb24ucGFycXVldCIpLAogICAgKGYie0hGfS9xYW5hc3Rlay9Ib0MvcGFycXVldC9Ib0MvdGVzdC8wLnBhcnF1ZXQiLCAiaG9jL3Rlc3QucGFycXVldCIpLAogICAgIyBHZW5lcmFsLWRvbWFpbiByZWZlcmVuY2UgY29ycHVzIGZvciB0aGUgRFJJRlQgY29udHJhc3QuCiAgICAoZiJ7SEZ9L1NhbGVzZm9yY2Uvd2lraXRleHQvcGFycXVldC93aWtpdGV4dC0xMDMtcmF3LXYxL3ZhbGlkYXRpb24vMC5wYXJxdWV0IiwKICAgICAicmVmZXJlbmNlL3dpa2l0ZXh0X3ZhbC5wYXJxdWV0IiksCiAgICAoZiJ7SEZ9L1NhbGVzZm9yY2Uvd2lraXRleHQvcGFycXVldC93aWtpdGV4dC0xMDMtcmF3LXYxL3Rlc3QvMC5wYXJxdWV0IiwKICAgICAicmVmZXJlbmNlL3dpa2l0ZXh0X3Rlc3QucGFycXVldCIpLApdCgoKZGVmIG1haW4oKToKICAgIG9rID0gVHJ1ZQogICAgZm9yIHVybCwgcmVsIGluIEZJTEVTOgogICAgICAgIGRzdCA9IG9zLnBhdGguam9pbihEQVRBLCByZWwpCiAgICAgICAgb3MubWFrZWRpcnMob3MucGF0aC5kaXJuYW1lKGRzdCksIGV4aXN0X29rPVRydWUpCiAgICAgICAgaWYgb3MucGF0aC5leGlzdHMoZHN0KSBhbmQgb3MucGF0aC5nZXRzaXplKGRzdCkgPiAwOgogICAgICAgICAgICBwcmludChmImhhdmUgIHtyZWx9IikKICAgICAgICAgICAgY29udGludWUKICAgICAgICBwcmludChmImdldCAgIHtyZWx9IC4uLiAiLCBlbmQ9IiIsIGZsdXNoPVRydWUpCiAgICAgICAgIyB0aGUgSHViIGludGVybWl0dGVudGx5IGFuc3dlcnMgNTAzOyByZXRyeSB3aXRoIGJhY2tvZmYgYmVmb3JlIGdpdmluZyB1cAogICAgICAgIGZvciBhdHRlbXB0IGluIHJhbmdlKDYpOgogICAgICAgICAgICB0cnk6CiAgICAgICAgICAgICAgICByZXEgPSB1cmxsaWIucmVxdWVzdC5SZXF1ZXN0KHVybCwgaGVhZGVycz17IlVzZXItQWdlbnQiOiAiTW96aWxsYS81LjAifSkKICAgICAgICAgICAgICAgIHdpdGggdXJsbGliLnJlcXVlc3QudXJsb3BlbihyZXEsIHRpbWVvdXQ9MTgwKSBhcyByOgogICAgICAgICAgICAgICAgICAgIGJvZHkgPSByLnJlYWQoKQogICAgICAgICAgICAgICAgd2l0aCBvcGVuKGRzdCwgIndiIikgYXMgZjoKICAgICAgICAgICAgICAgICAgICBmLndyaXRlKGJvZHkpCiAgICAgICAgICAgICAgICBwcmludChmIntvcy5wYXRoLmdldHNpemUoZHN0KS8xZTY6LjJmfSBNQiIpCiAgICAgICAgICAgICAgICBicmVhawogICAgICAgICAgICBleGNlcHQgRXhjZXB0aW9uIGFzIGU6ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICMgbm9xYTogQkxFMDAxCiAgICAgICAgICAgICAgICBpZiBhdHRlbXB0ID09IDU6CiAgICAgICAgICAgICAgICAgICAgb2sgPSBGYWxzZQogICAgICAgICAgICAgICAgICAgIHByaW50KCJGQUlMRUQ6IiwgZSkKICAgICAgICAgICAgICAgIGVsc2U6CiAgICAgICAgICAgICAgICAgICAgd2FpdCA9IDE1ICogMiAqKiBhdHRlbXB0CiAgICAgICAgICAgICAgICAgICAgcHJpbnQoZiJyZXRyeSBpbiB7d2FpdH1zICh7ZX0pIC4uLiAiLCBlbmQ9IiIsIGZsdXNoPVRydWUpCiAgICAgICAgICAgICAgICAgICAgdGltZS5zbGVlcCh3YWl0KQogICAgaWYgbm90IG9rOgogICAgICAgIHN5cy5leGl0KDEpCiAgICBwcmludCgiYWxsIGRhdGEgcHJlc2VudCIpCgoKaWYgX19uYW1lX18gPT0gIl9fbWFpbl9fIjoKICAgIG1haW4oKQo="}''')
for name, b64 in PAYLOAD.items():
    with open(os.path.join(WORK, 'src', name), 'wb') as f:
        f.write(base64.b64decode(b64))
print('wrote', len(PAYLOAD), 'modules:', sorted(PAYLOAD))


## Fetch datasets

In [ ]:
subprocess.run([sys.executable, 'src/download_data.py'], cwd=WORK, check=True)


## Restore results from earlier sessions
Kaggle: any attached `drift_results.zip` (a previous version's output) is unpacked. Colab: `runs/results` is linked to Google Drive, so earlier results are already there.

In [ ]:
import glob, zipfile, shutil
os.makedirs('runs/profiles', exist_ok=True)
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive/drift_results'
    os.makedirs(os.path.join(DRIVE, 'results'), exist_ok=True)
    if os.path.isdir('runs/results') and not os.path.islink('runs/results'):
        for p in glob.glob('runs/results/*.json'):
            shutil.copy(p, os.path.join(DRIVE, 'results'))
        shutil.rmtree('runs/results')
    if not os.path.exists('runs/results'):
        os.symlink(os.path.join(DRIVE, 'results'), 'runs/results')
os.makedirs('runs/results', exist_ok=True)
restored = 0
for z in sorted(glob.glob('/kaggle/input/**/drift_results.zip', recursive=True)):
    with zipfile.ZipFile(z) as zf:
        for n in zf.namelist():
            if n.startswith('results/') and n.endswith('.json'):
                dst = os.path.join('runs', n)
                if not os.path.exists(dst):
                    with zf.open(n) as s, open(dst, 'wb') as d:
                        d.write(s.read())
                    restored += 1
    print('restored from', z)
# a zip uploaded as a Kaggle Dataset arrives already unpacked
for p in glob.glob('/kaggle/input/**/results/*.json', recursive=True):
    dst = os.path.join('runs/results', os.path.basename(p))
    if not os.path.exists(dst):
        shutil.copy(p, dst)
        restored += 1
print('restored', restored, 'new results;',
      len(glob.glob('runs/results/*.json')), 'results present in total')


## Configure

`DRIFT_AMP=1` turns on fp16 autocast, a large speed-up on T4/P100, applied identically to every method so comparisons stay matched. Runs are split across all visible GPUs (two on Kaggle's T4 x2). No new run starts after `DEADLINE_HOURS`; the margin leaves room for the last run to finish.

In [ ]:
os.environ['DRIFT_AMP'] = '1'
# fail a stalled checkpoint download instead of hanging on it
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '60'
os.environ['HF_HUB_ETAG_TIMEOUT'] = '60'
MODEL = 'roberta-base'
DECODER = 'HuggingFaceTB/SmolLM2-360M'   # decoder SLM for the generality check
DEADLINE_HOURS = 10.5          # Kaggle kills a session at 12 h
DEADLINE = SESSION_START + DEADLINE_HOURS * 3600
NEED_PROFILES = True
os.makedirs('logs', exist_ok=True)

def snapshot(label=''):
    """Zip the result JSONs and profile summaries (not the large .pt
    profile tensors, which are recomputed cheaply)."""
    paths = sorted(glob.glob('runs/results/*.json')) + \
            sorted(glob.glob('runs/profiles/*.json'))
    out = os.path.join(WORK, 'drift_results.zip')
    with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
        for p in paths:
            z.write(p, os.path.relpath(p, 'runs'))
    if ON_COLAB:
        shutil.copy(out, DRIVE)
    nres = sum(1 for p in paths if 'results' in p)
    print(f'[snapshot {label}] {nres} results -> {out}', flush=True)

def run_plan(plan, seeds='1,2,3', model=MODEL):
    """Run one experiment plan, one worker per GPU, then snapshot."""
    if time.time() > DEADLINE:
        print(f'[{plan}] skipped: session deadline reached. Start a new '
              'session to continue.')
        return
    base = [sys.executable, 'src/grid.py', '--plan', plan,
            '--model', model, '--seeds', seeds]
    if NEED_PROFILES:
        subprocess.run(base + ['--profiles_only', '--deadline', str(DEADLINE)],
                       check=False)
    procs, logs = [], []
    for g in range(NGPU):
        log = f'logs/{plan}_gpu{g}.log'
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
        procs.append(subprocess.Popen(
            base + ['--shard', f'{g}/{NGPU}', '--deadline', str(DEADLINE),
                    '--no_profiles'],
            env=env, stdout=open(log, 'w'), stderr=subprocess.STDOUT))
        logs.append(log)
    t0 = time.time()
    while any(p.poll() is None for p in procs):
        time.sleep(120)
        done = sum(open(l, errors='ignore').read().count('\nDONE ') for l in logs)
        print(f'[{plan}] {(time.time()-t0)/60:.0f} min, {done} runs finished '
              f'this session', flush=True)
    for l in logs:
        txt = open(l, errors='ignore').read()
        print(f'--- tail of {l} ---')
        print(txt[-1500:])
        if 'Traceback' in txt:
            print(f'!! errors in {l}: search it for Traceback')
    snapshot(plan)


## Decoder smoke test
Profiles the decoder SLM and runs every compared method for one short epoch, to exercise the decoder code path (padding, module discovery, classification head) before committing GPU hours.

In [ ]:
import json, glob
def sh(*a):
    r = subprocess.run([sys.executable, *a], capture_output=True, text=True)
    tail = (r.stdout + r.stderr).strip().splitlines()[-8:]
    print('$', ' '.join(a[:6]), '->', 'OK' if r.returncode == 0 else f'EXIT {r.returncode}')
    print('   ' + '\n   '.join(tail))
    return r.returncode
fails = 0
t0 = time.time()
fails += sh('src/profile_drift.py', '--model', DECODER, '--task', 'chemprot') != 0
print(f'profile {time.time()-t0:.0f}s', flush=True)
for method in ('lora', 'eva', 'eva_white', 'drift'):
    t0 = time.time()
    fails += sh('src/run.py', '--model', DECODER, '--task', 'chemprot', '--method', method,
                '--epochs', '1', '--max_train', '600', '--lr', '3e-4', '--amp',
                '--tag', 'smoke') != 0
    print(f'{method} {time.time()-t0:.0f}s', flush=True)
for p in sorted(glob.glob('runs/results/*SmolLM2*smoke*.json')):
    r = json.load(open(p)); a = r['args']; res = r['result']
    print(a['method'], 'dev', round(res['dev_best']['micro_f1'], 4),
          'test', round(res['test']['micro_f1'], 4), 'adapter params',
          res.get('params_adapter'), 'modules', r['budget'].get('n_modules'),
          'train_s', round(res.get('train_time_s', 0)), 'ranks', r['rank_hist'])
print('DECODER SMOKE', 'PASSED' if fails == 0 else f'FAILED ({fails} failures)')
snapshot('decoder_smoke')


## Collect results
Download **drift_results.zip** (Kaggle: Output panel; Colab: `MyDrive/drift_results/`).

In [ ]:
snapshot('final')
subprocess.run([sys.executable, 'src/analyze.py', '--model', MODEL, '--dump'],
               check=False)
